# MetaMIRAGE Cumulative Qdrant Preload Builder

This notebook is the **offline orchestration pipeline** for building the cumulative MetaMIRAGE Qdrant base database one U.S. state at a time.

It implements the finalized architecture:

`Source Discovery → Extract/Normalize → Canonical Store + SQLite Ledger → Global Dedup → Qualification → Crop Dictionary → RAG Chunking → Metadata Enrichment + Hard Contract Validation → Batch Embedding → Batch Qdrant Upsert → Retry → Validation → Snapshot + Manifest`

### Hard invariants

- Successfully extracted content is persisted once in the canonical store.
- Completed work is never intentionally reprocessed on notebook rerun.
- Duplicate canonical documents are skipped globally across state runs.
- Qualification chunks and RAG chunks are separate.
- `tag != "msc"` is accepted.
- RAG chunking matches the runtime ingestion contract.
- No chunk can reach embedding/Qdrant until its metadata contract validates.
- Qdrant point IDs are deterministic.
- Failed work is retried inside the **current state run** and then becomes `permanently_failed`.
- A state is complete only after all work reaches a terminal state.
- Each completed state creates a cumulative Qdrant snapshot and manifest, and atomically updates the same cumulative `crop_occurrences.json` for that state's crop dictionary.

> The runtime/inference database lifecycle is intentionally out of scope. This notebook builds the offline cumulative base collection only.

## 1. Install dependencies

Delta's shared Python environment may be read-only, so packages are installed into a local `.python_packages` directory beside the notebook. The cell deliberately avoids reinstalling PyTorch.

In [ ]:
import sys
from pathlib import Path

PKG_DIR = Path.cwd() / ".python_packages"
PKG_DIR.mkdir(exist_ok=True)

!{sys.executable} -m pip install --no-cache-dir --target "{PKG_DIR}" \
    "huggingface-hub>=0.33.5,<1.0" \
    "transformers>=4.41,<5.0" \
    accelerate \
    bitsandbytes \
    sentence-transformers \
    pdfplumber \
    trafilatura \
    beautifulsoup4 \
    requests \
    tqdm \
    readability-lxml \
    lxml_html_clean \
    qdrant-client==1.18.0 \
    zstandard \
    openpyxl \
    xlrd>=2.0.1 \
    python-dateutil

# Remove local sympy/mpmath copies if pip pulled them into the target directory.
# Delta's shared PyTorch environment carries compatible versions and should own these.
import shutil
for pattern in ["sympy", "sympy-*", "mpmath", "mpmath-*"]:
    for candidate in PKG_DIR.glob(pattern):
        if candidate.is_dir():
            shutil.rmtree(candidate, ignore_errors=True)
        else:
            candidate.unlink(missing_ok=True)

if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))

print("Local package directory:", PKG_DIR)

## 2. Imports and run configuration

In [ ]:
import csv
import gc
import gzip
import hashlib
import io
import json
import logging
import os
import re
import shutil
import sqlite3
import threading
import time
import traceback
import uuid
import zipfile

from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Optional, Sequence, Set, Tuple
from urllib.parse import urlparse, unquote

import requests
import torch
import zstandard as zstd
import pdfplumber
import trafilatura
import openpyxl
import xlrd

from bs4 import BeautifulSoup
from dateutil import parser as date_parser
from huggingface_hub import login
from readability import Document
from requests.adapters import HTTPAdapter
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from urllib3.util.retry import Retry

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    PayloadSchemaType,
    PointStruct,
    VectorParams,
)

logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)
logging.getLogger("trafilatura").setLevel(logging.ERROR)

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())

In [ ]:
# ============================================================
# EDIT THIS CELL FOR EACH STATE RUN
# ============================================================

# The complete builder directory can be moved anywhere.
# Open/run this notebook from that directory; all file locations are relative.
BASE_DIR = Path.cwd().resolve()

BUILD_ID = "build_2026_08"
STATE_NAME = "Illinois"
STATE_CODE = "IL"
RUN_SEQUENCE = 1

RUN_ID = f"{RUN_SEQUENCE:03d}_{STATE_CODE.upper()}"

# ---------- Inputs ----------
# Inputs are AUTO-DISCOVERED directly under BASE_DIR.
#
# Filename contracts are case-insensitive:
#   *PDF*.zip                    -> the single PDF ZIP
#   *CSV*.zip                    -> the single CSV ZIP
#   *URL*.txt/.xlsx/.xlsm/.xls  -> the single URL input file
#
# At most one matching file of each type may exist at a time.
# PDF/CSV ZIPs are unpacked automatically.
URL_FILE: Optional[Path] = None
PDF_DIR: Optional[Path] = None
PDF_ZIP_FILE: Optional[Path] = None
CSV_ZIP_FILE: Optional[Path] = None
CSV_INPUTS: List[Dict[str, Any]] = []

# Fixed support files relative to BASE_DIR.
CROP_OCCURRENCE_JSON: Optional[Path] = BASE_DIR / "crop_occurrences.json"
HARDINESS_CSV: Path = BASE_DIR / "county_state_hardiness_zone.csv"

# ---------- Qdrant ----------
QDRANT_URL = "http://127.0.0.1:6333"
QDRANT_API_KEY: Optional[str] = None
QDRANT_COLLECTION = "mirage_base_build"

# For RUN_SEQUENCE > 1, point this at the immediately previous run folder.
# If the live Qdrant collection is absent/inconsistent, the notebook can restore
# the prior cumulative snapshot automatically.
PREVIOUS_RUN_DIR: Optional[Path] = None
AUTO_RESTORE_PREVIOUS_SNAPSHOT = True

# ---------- Models ----------
CLASSIFIER_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

# Prefer environment variable HF_TOKEN. If absent, model-loading cell prompts.
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()

# ---------- Qualification ----------
QUALIFICATION_CHUNK_CHARS = 7000
QUALIFICATION_OVERLAP_CHARS = 700
MAX_QUALIFICATION_CHUNKS = 20
EARLY_STOP_TOTAL_ENTITIES = 5

# Initial attempt + up to 2 retries = max 3 attempts.
MAX_STAGE_RETRIES = 2

# ---------- Runtime-compatible RAG chunking ----------
RAG_CHUNK_SIZE = 480
RAG_CHUNK_OVERLAP = 80
RAG_HARD_CAP = 512

# ---------- Batching / concurrency ----------
EXTRACTION_WORKERS = 8
EMBED_BATCH_SIZE = 64
QDRANT_UPSERT_BATCH_SIZE = 128

# ---------- Versioned cache contracts ----------
EXTRACTOR_VERSION = "canonical-extractor-v1"
CLASSIFIER_VERSION = "llama-qualification-v1"
CHUNKER_VERSION = "bge-480-80-v1"
METADATA_CONTRACT_VERSION = "runtime-canonical-v1"

# ---------- Safety / development ----------
# None = process every discovered source.
DEBUG_SOURCE_LIMIT: Optional[int] = None

# Keep False until the configuration above has been reviewed.
RUN_PIPELINE = True

# Sample retrieval checks before snapshot finalization.
VALIDATION_QUERIES = [
    "crop disease management",
    "agricultural pest control",
]

print("RUN_ID:", RUN_ID)
print("Base directory:", BASE_DIR)

## 3. Directory layout and configuration validation

The notebook keeps a **global** SQLite ledger and canonical store across all 50 state runs. Run-specific artifacts live under `runs/<BUILD_ID>/<RUN_ID>/`.

In [ ]:
STATE_KEY = re.sub(r"\s+", " ", STATE_NAME.strip().lower())
STATE_CODE = STATE_CODE.strip().upper()

STATE_DIR = BASE_DIR / "runs" / BUILD_ID / RUN_ID
CANONICAL_DIR = BASE_DIR / "canonical"
STATE_DB_PATH = BASE_DIR / "pipeline_state.db"
QDRANT_LOCAL_DIR = BASE_DIR / "qdrant"

for path in [BASE_DIR, STATE_DIR, CANONICAL_DIR, QDRANT_LOCAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def _existing_optional(path: Optional[Path]) -> Optional[Path]:
    if path is None:
        return None
    path = Path(path)
    return path if path.exists() else None

def _find_single_input(keyword: str, suffixes: Sequence[str]) -> Optional[Path]:
    keyword = keyword.lower()
    suffixes = {suffix.lower() for suffix in suffixes}

    candidates = sorted(
        [
            path
            for path in BASE_DIR.iterdir()
            if path.is_file()
            and keyword in path.name.lower()
            and path.suffix.lower() in suffixes
        ],
        key=lambda path: path.name.lower(),
    )

    if len(candidates) > 1:
        raise RuntimeError(
            f"Expected at most one {keyword.upper()} input under {BASE_DIR}, "
            f"but found {len(candidates)}: {[p.name for p in candidates]}"
        )

    return candidates[0] if candidates else None

# Auto-discover the current run inputs.
PDF_ZIP_FILE = _find_single_input("PDF", {".zip"})
CSV_ZIP_FILE = _find_single_input("CSV", {".zip"})
URL_FILE = _find_single_input("URL", {".txt", ".xlsx", ".xlsm", ".xls"})
PDF_DIR = None

# Extract every CSV in the single CSV ZIP, if present.
CSV_INPUTS = []
if CSV_ZIP_FILE is not None:
    csv_extracted_dir = STATE_DIR / "_csv_zip_extracted"
    csv_extracted_dir.mkdir(parents=True, exist_ok=True)
    marker = csv_extracted_dir / ".extracted_complete"

    if not marker.exists():
        with zipfile.ZipFile(CSV_ZIP_FILE, "r") as zf:
            zf.extractall(csv_extracted_dir)
        marker.write_text(
            datetime.now(timezone.utc).isoformat(),
            encoding="utf-8",
        )

    csv_paths = sorted(
        path
        for path in csv_extracted_dir.rglob("*")
        if path.is_file() and path.suffix.lower() == ".csv"
    )

    if not csv_paths:
        raise RuntimeError(f"No CSV files found inside {CSV_ZIP_FILE}")

    # Existing generic CSV logic auto-detects common title/date/url/county fields.
    CSV_INPUTS = [{"path": path} for path in csv_paths]

CROP_OCCURRENCE_JSON = _existing_optional(CROP_OCCURRENCE_JSON)
PREVIOUS_RUN_DIR = _existing_optional(PREVIOUS_RUN_DIR)

if not HARDINESS_CSV.exists():
    raise FileNotFoundError(f"Hardiness mapping not found: {HARDINESS_CSV}")

for cfg in CSV_INPUTS:
    cfg["path"] = Path(cfg["path"])
    if not cfg["path"].exists():
        raise FileNotFoundError(f"CSV input not found: {cfg['path']}")

if RUN_SEQUENCE == 1 and PREVIOUS_RUN_DIR is not None:
    print("Warning: PREVIOUS_RUN_DIR is ignored for sequence 1.")

if RUN_SEQUENCE > 1 and PREVIOUS_RUN_DIR is None:
    print(
        "WARNING: this is a cumulative run > 1 but PREVIOUS_RUN_DIR is None. "
        "This is okay only if the live Qdrant collection already contains the prior cumulative state."
    )

print("Run output directory:", STATE_DIR)
print("Ledger:", STATE_DB_PATH)
print("Canonical store:", CANONICAL_DIR)
print("URL file:", URL_FILE)
print("PDF zip:", PDF_ZIP_FILE)
print("CSV zip:", CSV_ZIP_FILE)
print("CSV inputs:", [str(x["path"]) for x in CSV_INPUTS])
print("Crop occurrence JSON:", CROP_OCCURRENCE_JSON)

## 4. SQLite processing ledger

`source_tasks` is an operational addition needed before a canonical `document_id` exists. After successful extraction, the durable document-level state lives in `documents` / `run_documents`.

SQLite does **not** hold full extracted documents or RAG chunk text.

In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def get_db() -> sqlite3.Connection:
    conn = sqlite3.connect(STATE_DB_PATH, timeout=60)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("PRAGMA busy_timeout = 60000")
    conn.execute("PRAGMA synchronous = NORMAL")
    return conn

SCHEMA_SQL = r"""
CREATE TABLE IF NOT EXISTS runs (
    run_id TEXT PRIMARY KEY,
    build_id TEXT NOT NULL,
    sequence_number INTEGER NOT NULL,
    state_name TEXT NOT NULL,
    state_code TEXT NOT NULL,
    status TEXT NOT NULL,
    started_at TEXT,
    completed_at TEXT,
    max_stage_retries INTEGER NOT NULL DEFAULT 2,
    snapshot_path TEXT,
    manifest_path TEXT,
    error TEXT
);

CREATE TABLE IF NOT EXISTS source_tasks (
    source_key TEXT PRIMARY KEY,
    run_id TEXT NOT NULL,
    source_type TEXT NOT NULL,
    source_uri TEXT NOT NULL,
    source_payload_json TEXT,
    status TEXT NOT NULL,
    attempt_count INTEGER NOT NULL DEFAULT 0,
    document_id TEXT,
    duplicate INTEGER NOT NULL DEFAULT 0,
    last_error TEXT,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY(run_id) REFERENCES runs(run_id)
);

CREATE INDEX IF NOT EXISTS idx_source_tasks_run_status
ON source_tasks(run_id, status);

CREATE TABLE IF NOT EXISTS documents (
    document_id TEXT PRIMARY KEY,
    content_hash TEXT NOT NULL UNIQUE,
    raw_hash TEXT,
    source_type TEXT NOT NULL,
    canonical_text_path TEXT NOT NULL,
    canonical_metadata_path TEXT NOT NULL,
    canonical_text_chars INTEGER,
    canonical_text_bytes INTEGER,
    language TEXT,
    extractor_version TEXT NOT NULL,
    extraction_status TEXT NOT NULL,
    first_seen_run_id TEXT NOT NULL,
    first_seen_state TEXT NOT NULL,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY(first_seen_run_id) REFERENCES runs(run_id)
);

CREATE INDEX IF NOT EXISTS idx_documents_raw_hash
ON documents(raw_hash);

CREATE TABLE IF NOT EXISTS run_documents (
    run_id TEXT NOT NULL,
    document_id TEXT NOT NULL,
    source_type TEXT NOT NULL,
    source_uri TEXT,
    discovery_order INTEGER,
    duplicate INTEGER NOT NULL DEFAULT 0,
    document_status TEXT NOT NULL,
    qualification_status TEXT,
    accepted INTEGER,
    qualification_tag TEXT,
    qualification_subtags_json TEXT,
    qualification_entities_json TEXT,
    qualification_reason TEXT,
    classifier_version TEXT,
    rag_status TEXT,
    qdrant_status TEXT,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    PRIMARY KEY(run_id, document_id),
    FOREIGN KEY(run_id) REFERENCES runs(run_id),
    FOREIGN KEY(document_id) REFERENCES documents(document_id)
);

CREATE INDEX IF NOT EXISTS idx_run_documents_run_status
ON run_documents(run_id, document_status);

CREATE TABLE IF NOT EXISTS qualification_chunks (
    qualification_chunk_id TEXT PRIMARY KEY,
    run_id TEXT NOT NULL,
    document_id TEXT NOT NULL,
    chunk_index INTEGER NOT NULL,
    chunk_hash TEXT NOT NULL,
    status TEXT NOT NULL,
    classification_result_json TEXT,
    attempt_count INTEGER NOT NULL DEFAULT 0,
    last_error TEXT,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY(run_id) REFERENCES runs(run_id),
    FOREIGN KEY(document_id) REFERENCES documents(document_id),
    UNIQUE(run_id, document_id, chunk_index)
);

CREATE INDEX IF NOT EXISTS idx_qchunks_run_status
ON qualification_chunks(run_id, status);

CREATE TABLE IF NOT EXISTS rag_chunks (
    rag_chunk_id TEXT PRIMARY KEY,
    run_id TEXT NOT NULL,
    document_id TEXT NOT NULL,
    page INTEGER NOT NULL,
    chunk_index INTEGER NOT NULL,
    chunk_hash TEXT NOT NULL,
    token_count INTEGER NOT NULL,
    chunker_version TEXT NOT NULL,
    metadata_json TEXT,
    status TEXT NOT NULL,
    embedding_status TEXT NOT NULL,
    qdrant_status TEXT NOT NULL,
    metadata_attempt_count INTEGER NOT NULL DEFAULT 0,
    embedding_attempt_count INTEGER NOT NULL DEFAULT 0,
    qdrant_attempt_count INTEGER NOT NULL DEFAULT 0,
    failure_stage TEXT,
    last_error TEXT,
    qdrant_point_id TEXT NOT NULL,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY(run_id) REFERENCES runs(run_id),
    FOREIGN KEY(document_id) REFERENCES documents(document_id),
    UNIQUE(document_id, page, chunk_index, chunker_version)
);

CREATE INDEX IF NOT EXISTS idx_rag_chunks_run_status
ON rag_chunks(run_id, status);

CREATE INDEX IF NOT EXISTS idx_rag_chunks_hash
ON rag_chunks(chunk_hash);

CREATE TABLE IF NOT EXISTS attempts (
    attempt_id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id TEXT NOT NULL,
    unit_type TEXT NOT NULL,
    unit_id TEXT NOT NULL,
    stage TEXT NOT NULL,
    attempt_number INTEGER NOT NULL,
    status TEXT NOT NULL,
    started_at TEXT NOT NULL,
    completed_at TEXT,
    error_type TEXT,
    error_message TEXT
);

CREATE INDEX IF NOT EXISTS idx_attempts_run
ON attempts(run_id);

CREATE TABLE IF NOT EXISTS snapshots (
    snapshot_id TEXT PRIMARY KEY,
    build_id TEXT NOT NULL,
    run_id TEXT NOT NULL,
    collection_name TEXT NOT NULL,
    snapshot_path TEXT NOT NULL,
    manifest_path TEXT NOT NULL,
    checksum_sha256 TEXT NOT NULL,
    qdrant_point_count INTEGER NOT NULL,
    created_at TEXT NOT NULL,
    FOREIGN KEY(run_id) REFERENCES runs(run_id)
);

CREATE TABLE IF NOT EXISTS classification_cache (
    cache_key TEXT PRIMARY KEY,
    model_id TEXT NOT NULL,
    classifier_version TEXT NOT NULL,
    state_key TEXT NOT NULL,
    crop_hash TEXT NOT NULL,
    content_hash TEXT NOT NULL,
    validated_output_json TEXT NOT NULL,
    raw_output TEXT,
    created_at TEXT NOT NULL
);
"""

with get_db() as conn:
    conn.execute("PRAGMA journal_mode = WAL")
    conn.executescript(SCHEMA_SQL)

    # Small forward migration so rerunning a newer notebook against an existing ledger is safe.
    rag_columns = {row[1] for row in conn.execute("PRAGMA table_info(rag_chunks)").fetchall()}
    if "metadata_attempt_count" not in rag_columns:
        conn.execute(
            "ALTER TABLE rag_chunks ADD COLUMN metadata_attempt_count INTEGER NOT NULL DEFAULT 0"
        )

    existing = conn.execute("SELECT * FROM runs WHERE run_id=?", (RUN_ID,)).fetchone()
    if existing is None:
        conn.execute(
            """
            INSERT INTO runs(
                run_id, build_id, sequence_number, state_name, state_code,
                status, started_at, max_stage_retries
            ) VALUES (?, ?, ?, ?, ?, 'pending', ?, ?)
            """,
            (
                RUN_ID, BUILD_ID, RUN_SEQUENCE, STATE_NAME, STATE_CODE,
                utc_now(), MAX_STAGE_RETRIES,
            ),
        )
    else:
        if existing["build_id"] != BUILD_ID or existing["state_code"] != STATE_CODE:
            raise RuntimeError(
                f"RUN_ID {RUN_ID} already exists with different build/state metadata."
            )

print("SQLite schema ready.")

## 5. Common hashing, canonical-store, and attempt helpers

In [ ]:
UUID_NAMESPACE = uuid.UUID("8d36fffe-8bed-4f98-9d6e-8a53e2755a91")

def normalize_hash_text(text: str) -> str:
    return " ".join((text or "").lower().split())

def stable_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()

def content_hash(text: str) -> str:
    return stable_hash(normalize_hash_text(text))

def bytes_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def canonical_paths(document_id: str) -> Tuple[Path, Path]:
    shard = document_id[:2]
    doc_dir = CANONICAL_DIR / shard / document_id
    return doc_dir / "content.txt.zst", doc_dir / "metadata.json"

CANONICAL_PERSIST_LOCK = threading.Lock()

def write_zstd_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".{uuid.uuid4().hex}.tmp")
    compressor = zstd.ZstdCompressor(level=6)
    try:
        with open(tmp, "wb") as raw:
            with compressor.stream_writer(raw) as writer:
                writer.write(text.encode("utf-8"))
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)

def read_zstd_text(path: Path) -> str:
    decompressor = zstd.ZstdDecompressor()
    with open(path, "rb") as raw:
        with decompressor.stream_reader(raw) as reader:
            return reader.read().decode("utf-8")

def write_json_atomic(path: Path, obj: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)

def load_json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))

def deterministic_point_uuid(chunk_id: str) -> str:
    return str(uuid.uuid5(UUID_NAMESPACE, chunk_id))

def record_attempt(
    unit_type: str,
    unit_id: str,
    stage: str,
    attempt_number: int,
    status: str,
    started_at: str,
    error: Optional[BaseException] = None,
):
    error_type = type(error).__name__ if error else None
    error_message = str(error)[:4000] if error else None
    with get_db() as conn:
        conn.execute(
            """
            INSERT INTO attempts(
                run_id, unit_type, unit_id, stage, attempt_number,
                status, started_at, completed_at, error_type, error_message
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                RUN_ID, unit_type, unit_id, stage, attempt_number,
                status, started_at, utc_now(), error_type, error_message,
            ),
        )

print("Persistence helpers ready.")

## 6. Hardiness-zone contract

This independently reconstructs the inference mapping contract from `county_state_hardiness_zone.csv`.

Resolution order:

1. `(county, state) → exact zone`
2. otherwise `state → modal zone`
3. unrecognized state → `""`

For Alaska and Hawaii only, `hardiness_zone=""` is an explicitly allowed metadata-contract exception.

In [ ]:
def normalize_token(value: Any) -> str:
    value = re.sub(r"[^a-z0-9\s]", " ", str(value or "").lower())
    return re.sub(r"\s+", " ", value).strip()

def normalize_county(value: Any) -> str:
    county = normalize_token(value)
    return re.sub(r"\bcounty\b$", "", county).strip()

def load_hardiness_lookup(csv_path: Path):
    county_lookup: Dict[Tuple[str, str], str] = {}
    state_zone_counts: Dict[str, Counter] = defaultdict(Counter)
    abbr_to_name: Dict[str, str] = {}

    with open(csv_path, newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            state_name = re.sub(r"\s+", " ", (row.get("state") or "").strip())
            state_abbr = (row.get("state_abbr") or "").strip().upper()
            county = normalize_county(row.get("county", ""))
            zone = (row.get("hardiness_zone") or "").strip()

            if state_name and state_abbr:
                abbr_to_name[state_abbr] = state_name.upper()

            if not state_name:
                continue

            canonical_name = state_name.upper()

            if county and zone:
                county_lookup[(county, canonical_name)] = zone

            if zone:
                state_zone_counts[canonical_name][zone] += 1

    state_modal_lookup = {
        state: counts.most_common(1)[0][0]
        for state, counts in state_zone_counts.items()
        if counts
    }

    name_to_abbr = {name: abbr for abbr, name in abbr_to_name.items()}
    return county_lookup, state_modal_lookup, abbr_to_name, name_to_abbr

(
    COUNTY_ZONE_LOOKUP,
    STATE_MODAL_ZONE_LOOKUP,
    STATE_ABBR_TO_NAME,
    STATE_NAME_TO_ABBR,
) = load_hardiness_lookup(HARDINESS_CSV)

def canonical_state(value: Any) -> str:
    state = normalize_token(value).upper()
    if state in STATE_ABBR_TO_NAME:
        return STATE_ABBR_TO_NAME[state]
    if state in STATE_NAME_TO_ABBR:
        return state
    return ""

def hardiness_zone_for_location(location: str) -> str:
    if not location or not location.strip():
        return ""

    parts = [part.strip() for part in location.split(",") if part.strip()]
    if not parts:
        return ""

    if len(parts) >= 2:
        # Preferred: State, County
        state = canonical_state(parts[0])
        county = normalize_county(parts[1])
        if state:
            return (
                COUNTY_ZONE_LOOKUP.get((county, state))
                or STATE_MODAL_ZONE_LOOKUP.get(state, "")
            )

        # Legacy: County, State
        county = normalize_county(parts[0])
        state = canonical_state(parts[1])
        if state:
            return (
                COUNTY_ZONE_LOOKUP.get((county, state))
                or STATE_MODAL_ZONE_LOOKUP.get(state, "")
            )
        return ""

    state = canonical_state(parts[0])
    return STATE_MODAL_ZONE_LOOKUP.get(state, "")

resolved_state = canonical_state(STATE_CODE) or canonical_state(STATE_NAME)
if not resolved_state:
    raise ValueError(f"State is not recognized by hardiness mapping: {STATE_NAME} / {STATE_CODE}")

print("Canonical state:", resolved_state)
print("State modal zone:", hardiness_zone_for_location(STATE_NAME) or "<empty>")
if STATE_CODE in {"AK", "HI"}:
    print("AK/HI exception active: empty hardiness_zone is allowed.")

## 7. Crop occurrence input + crop dictionary helpers

In [ ]:
ALLOWED_TAGS = {"crops", "pest", "disease", "management", "multi", "msc"}
ENTITY_FIELDS = ["disease", "pests", "management"]

def normalize_name(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "").strip().lower())

def normalize_entity(value: Any) -> Optional[str]:
    if value is None:
        return None
    v = normalize_name(value)
    if len(v) < 3:
        return None
    generic_rejects = {
        "disease", "diseases", "pest", "pests", "issue", "issues",
        "problem", "problems", "management", "control", "crop", "crops",
        "plant", "plants", "unknown", "none", "n/a", "na"
    }
    return None if v in generic_rejects else v

def load_crop_occurrence_state(
    path: Optional[Path],
    state_key: str,
) -> Dict[str, Dict[str, Any]]:
    if path is None:
        print("No crop occurrence JSON configured; crop list will start empty.")
        return {}

    raw = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(raw, dict):
        raise ValueError("crop_occurrences.json must contain a top-level JSON object.")

    normalized_state_keys = {
        normalize_name(key): key
        for key in raw.keys()
    }

    if state_key not in normalized_state_keys:
        raise KeyError(
            f"State '{STATE_NAME}' not found in crop_occurrences.json. "
            f"Available keys include: {list(raw.keys())[:20]}"
        )

    state_payload = raw[normalized_state_keys[state_key]]
    if not isinstance(state_payload, dict):
        raise ValueError(
            f"State '{STATE_NAME}' in crop_occurrences.json must map to an object."
        )

    result: Dict[str, Dict[str, Any]] = {}

    for crop_name, entry in state_payload.items():
        crop = normalize_name(crop_name)
        if not crop:
            continue

        if not isinstance(entry, dict):
            entry = {}

        result[crop] = {
            "occurrence": int(entry.get("occurrence", 0) or 0),
            "disease": dict(entry.get("disease", {}) or {}),
            "pests": dict(entry.get("pests", {}) or {}),
            "management": dict(entry.get("management", {}) or {}),
        }

        if entry.get("added"):
            result[crop]["added"] = True

    return result

CROP_OCCURRENCE_STATE = load_crop_occurrence_state(
    CROP_OCCURRENCE_JSON,
    STATE_KEY,
)

CROP_OCCURRENCE_MAP = {
    crop: int(entry.get("occurrence", 0) or 0)
    for crop, entry in CROP_OCCURRENCE_STATE.items()
}
CROP_LIST = sorted(CROP_OCCURRENCE_MAP)

def initialize_crop_dictionary() -> Dict[str, Any]:
    return {
        STATE_KEY: {
            crop: {
                "disease": dict(entry.get("disease", {}) or {}),
                "pests": dict(entry.get("pests", {}) or {}),
                "management": dict(entry.get("management", {}) or {}),
                "occurrence": int(entry.get("occurrence", 0) or 0),
                **({"added": True} if entry.get("added") else {}),
            }
            for crop, entry in CROP_OCCURRENCE_STATE.items()
        }
    }

def build_crop_dictionary_from_ledger() -> Dict[str, Any]:
    crop_dict = initialize_crop_dictionary()

    with get_db() as conn:
        rows = conn.execute(
            """
            SELECT qualification_tag, qualification_entities_json
            FROM run_documents
            WHERE run_id=?
              AND duplicate=0
              AND qualification_status='succeeded'
              AND qualification_tag != 'msc'
            """,
            (RUN_ID,),
        ).fetchall()

    for row in rows:
        entities = json.loads(row["qualification_entities_json"] or "{}")

        for crop, fields in entities.items():
            crop = normalize_name(crop)
            if not crop:
                continue

            if crop not in crop_dict[STATE_KEY]:
                crop_dict[STATE_KEY][crop] = {
                    "disease": {},
                    "pests": {},
                    "management": {},
                    "occurrence": CROP_OCCURRENCE_MAP.get(crop, 0),
                    "added": True,
                }

            for field in ENTITY_FIELDS:
                unique_entities = {
                    ent
                    for ent in (
                        normalize_entity(x)
                        for x in fields.get(field, []) or []
                    )
                    if ent
                }

                for ent in unique_entities:
                    current = crop_dict[STATE_KEY][crop][field].get(ent, 0)
                    crop_dict[STATE_KEY][crop][field][ent] = current + 1

    for crop, entry in crop_dict[STATE_KEY].items():
        for field in ENTITY_FIELDS:
            entry[field] = dict(sorted(entry[field].items()))

    return crop_dict

def update_cumulative_crop_occurrences(
    crop_dictionary: Dict[str, Any],
) -> Path:
    if CROP_OCCURRENCE_JSON is None:
        raise RuntimeError(
            "CROP_OCCURRENCE_JSON must be configured to persist the cumulative crop dictionary."
        )

    path = Path(CROP_OCCURRENCE_JSON)
    raw = json.loads(path.read_text(encoding="utf-8"))

    if not isinstance(raw, dict):
        raise ValueError("crop_occurrences.json must contain a top-level JSON object.")

    normalized_state_keys = {
        normalize_name(key): key
        for key in raw.keys()
    }

    existing_key = normalized_state_keys.get(STATE_KEY, STATE_KEY)
    raw[existing_key] = crop_dictionary.get(STATE_KEY, {})

    write_json_atomic(path, raw)
    return path

print(f"Loaded {len(CROP_LIST)} crop names for qualification.")


## 8. Source extraction + canonicalization

Successful extraction returns one normalized canonical document:

- PDF → one canonical document, while preserving page boundaries in metadata.
- Web URL → one canonical document.
- CSV row → one canonical document.

`month_year` is populated only when a credible source date is determinable; otherwise it is `""`.

In [ ]:
_thread_local = threading.local()

JUNK_URL_PATTERNS = [
    "youtube.com",
    "youtu.be",
    "youtube-nocookie.com",
    "facebook.com",
    "instagram.com",
    "linkedin.com",
    "twitter.com",
    "x.com/",
]

def get_http_session() -> requests.Session:
    if not hasattr(_thread_local, "session"):
        session = requests.Session()
        retries = Retry(
            total=3,
            connect=3,
            read=3,
            status=3,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET", "HEAD"],
            raise_on_status=False,
        )
        adapter = HTTPAdapter(
            pool_connections=100,
            pool_maxsize=100,
            max_retries=retries,
            pool_block=False,
        )
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        _thread_local.session = session
    return _thread_local.session

def is_junk_url(url: str) -> bool:
    u = url.lower()
    return any(pattern in u for pattern in JUNK_URL_PATTERNS)

def clean_text_common(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    ref_match = re.search(
        r"\n\s*(references|bibliography|literature cited)\s*\n",
        text,
        flags=re.IGNORECASE,
    )
    if ref_match and ref_match.start() > len(text) * 0.55:
        text = text[:ref_match.start()]
    return text.strip()

def normalize_month_year(value: Any) -> str:
    raw = str(value or "").strip()
    if not raw:
        return ""

    m = re.search(r"\b(19|20)\d{2}[-/](0[1-9]|1[0-2])\b", raw)
    if m:
        return m.group(0).replace("/", "-")[:7]

    # PDF dates often look like D:20240115103000.
    m = re.search(r"(?:D:)?((?:19|20)\d{2})(0[1-9]|1[0-2])", raw)
    if m:
        return f"{m.group(1)}-{m.group(2)}"

    try:
        dt = date_parser.parse(raw, fuzzy=False)
        if 1900 <= dt.year <= datetime.now().year + 1:
            return f"{dt.year:04d}-{dt.month:02d}"
    except Exception:
        pass
    return ""

def extract_web_month_year(soup: BeautifulSoup) -> str:
    candidates: List[str] = []

    meta_keys = [
        ("property", "article:published_time"),
        ("property", "article:modified_time"),
        ("name", "date"),
        ("name", "pubdate"),
        ("name", "publishdate"),
        ("name", "publication_date"),
        ("name", "dc.date"),
        ("itemprop", "datePublished"),
        ("itemprop", "dateModified"),
    ]
    for attr, key in meta_keys:
        tag = soup.find("meta", attrs={attr: key})
        if tag and tag.get("content"):
            candidates.append(tag.get("content"))

    for tag in soup.find_all("time"):
        if tag.get("datetime"):
            candidates.append(tag.get("datetime"))

    # JSON-LD is often the cleanest publication date.
    for script in soup.find_all("script", attrs={"type": "application/ld+json"}):
        try:
            payload = json.loads(script.string or "")
        except Exception:
            continue

        stack = payload if isinstance(payload, list) else [payload]
        for obj in stack:
            if isinstance(obj, dict):
                for key in ["datePublished", "dateCreated", "dateModified"]:
                    if obj.get(key):
                        candidates.append(str(obj[key]))

    for candidate in candidates:
        normalized = normalize_month_year(candidate)
        if normalized:
            return normalized
    return ""

def fallback_title_from_url(url: str) -> str:
    parsed = urlparse(url)
    slug = unquote(parsed.path.rstrip("/").split("/")[-1] or parsed.netloc)
    slug = re.sub(r"[-_]+", " ", slug).strip()
    return slug or parsed.netloc or "Untitled web page"

def extract_url_document(url: str) -> Dict[str, Any]:
    if is_junk_url(url):
        raise ValueError("junk_url_prefilter")

    response = get_http_session().get(
        url,
        timeout=(8, 30),
        headers={"User-Agent": "Mozilla/5.0 (MetaMIRAGE offline research preload)"},
        allow_redirects=True,
    )
    response.raise_for_status()

    content_type = response.headers.get("Content-Type", "").lower()
    if "text/html" not in content_type:
        raise ValueError(f"non_html_content_type:{content_type}")

    html = response.text
    if not html or len(html) < 200:
        raise ValueError("empty_or_too_short_html")

    soup = BeautifulSoup(html, "lxml")

    title = ""
    if soup.title and soup.title.get_text(strip=True):
        title = soup.title.get_text(" ", strip=True)
    if not title:
        h1 = soup.find("h1")
        if h1:
            title = h1.get_text(" ", strip=True)
    title = re.sub(r"\s+", " ", title).strip() or fallback_title_from_url(url)

    month_year = extract_web_month_year(soup) or normalize_month_year(response.headers.get("Last-Modified"))

    try:
        readable_html = Document(html).summary()
    except Exception:
        readable_html = html

    extracted = trafilatura.extract(
        readable_html,
        include_comments=False,
        include_tables=False,
        include_links=False,
        include_images=False,
        favor_precision=True,
        deduplicate=True,
    )
    if not extracted:
        extracted = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=False,
            include_links=False,
            include_images=False,
        )

    text = clean_text_common(extracted or "")
    if len(text) < 100:
        raise ValueError("web_extraction_empty_or_too_short")

    return {
        "text": text,
        "title": title,
        "month_year": month_year,
        "url": response.url or url,
        "raw_hash": stable_hash(html),
        "pages": [{"page": -1, "start": 0, "end": len(text)}],
        "source_metadata": {
            "requested_url": url,
            "final_url": response.url,
            "http_last_modified": response.headers.get("Last-Modified", ""),
        },
    }

def extract_pdf_document(path: Path) -> Dict[str, Any]:
    raw_hash = bytes_sha256(path)
    page_records = []
    joined_parts: List[str] = []
    cursor = 0

    with pdfplumber.open(str(path)) as pdf:
        metadata = pdf.metadata or {}
        title = re.sub(r"\s+", " ", str(metadata.get("Title") or "")).strip()
        title = title or path.stem
        month_year = (
            normalize_month_year(metadata.get("CreationDate"))
            or normalize_month_year(metadata.get("ModDate"))
        )

        for page_index, page in enumerate(pdf.pages):
            text = clean_text_common(page.extract_text() or "")
            if not text:
                continue

            if joined_parts:
                cursor += 2  # "\n\n"
            start = cursor
            joined_parts.append(text)
            cursor += len(text)
            end = cursor

            page_records.append({
                "page": page_index,
                "start": start,
                "end": end,
            })

    full_text = "\n\n".join(joined_parts).strip()
    if len(full_text) < 100:
        raise ValueError("pdf_extraction_empty_or_too_short")

    return {
        "text": full_text,
        "title": title,
        "month_year": month_year,
        "url": "",
        "raw_hash": raw_hash,
        "pages": page_records,
        "source_metadata": {
            "original_filename": path.name,
            "page_count_with_text": len(page_records),
        },
    }

COMMON_DATE_FIELDS = [
    "month_year", "publication_date", "published_date",
    "publish_date", "date", "created_at", "updated_at",
]
COMMON_URL_FIELDS = ["url", "source_url", "link"]
COMMON_COUNTY_FIELDS = ["county", "county_name"]
COMMON_TITLE_FIELDS = ["title", "name", "crop_name", "entity_name"]

def first_nonempty(row: Dict[str, Any], fields: Sequence[str]) -> str:
    for field in fields:
        value = row.get(field)
        if value is not None and str(value).strip():
            return str(value).strip()
    return ""

def deterministic_csv_narrative(
    row: Dict[str, Any],
    include_fields: Optional[Sequence[str]] = None,
) -> str:
    if include_fields:
        keys = [k for k in include_fields if k in row]
    else:
        keys = sorted(row.keys(), key=lambda x: str(x).lower())

    lines = []
    for key in keys:
        value = row.get(key)
        if value is None or not str(value).strip():
            continue
        clean_key = re.sub(r"\s+", " ", str(key).strip())
        clean_value = re.sub(r"\s+", " ", str(value).strip())
        lines.append(f"{clean_key}: {clean_value}")

    return clean_text_common("\n".join(lines))

def extract_csv_row_document(
    row: Dict[str, Any],
    row_number: int,
    csv_path: Path,
    config: Dict[str, Any],
) -> Dict[str, Any]:
    text = deterministic_csv_narrative(row, config.get("include_fields"))
    if not text:
        raise ValueError("empty_csv_row")

    title_fields = config.get("title_fields") or COMMON_TITLE_FIELDS
    title_parts = [
        str(row.get(f)).strip()
        for f in title_fields
        if row.get(f) is not None and str(row.get(f)).strip()
    ]
    title = " - ".join(title_parts[:3])
    if not title:
        title = f"{STATE_NAME} {csv_path.stem} record {row_number}"

    date_field = config.get("date_field")
    raw_date = row.get(date_field) if date_field else first_nonempty(row, COMMON_DATE_FIELDS)
    month_year = normalize_month_year(raw_date)

    url_field = config.get("url_field")
    url = (
        str(row.get(url_field) or "").strip()
        if url_field
        else first_nonempty(row, COMMON_URL_FIELDS)
    )

    county_field = config.get("county_field")
    county = (
        str(row.get(county_field) or "").strip()
        if county_field
        else first_nonempty(row, COMMON_COUNTY_FIELDS)
    )

    canonical_raw = json.dumps(
        {str(k): "" if v is None else str(v) for k, v in row.items()},
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    )

    return {
        "text": text,
        "title": title,
        "month_year": month_year,
        "url": url,
        "raw_hash": stable_hash(canonical_raw),
        "pages": [{"page": -1, "start": 0, "end": len(text)}],
        "source_metadata": {
            "csv_file": str(csv_path),
            "csv_row_index": row_number,
            "county": county,
        },
    }

print("Extraction helpers ready.")

## 9. Source discovery and canonical-store ingestion

Discovery is idempotent. Successful extraction writes the canonical content and document metadata once. A fast raw-hash duplicate check is used for local PDFs/CSV rows when possible, followed by canonical-content deduplication for every source.

In [ ]:
def source_key_for(source_type: str, source_uri: str) -> str:
    return stable_hash(f"{RUN_ID}|{source_type}|{source_uri}")

def register_source_task(
    source_type: str,
    source_uri: str,
    payload: Optional[Dict[str, Any]] = None,
):
    key = source_key_for(source_type, source_uri)
    now = utc_now()
    with get_db() as conn:
        conn.execute(
            """
            INSERT OR IGNORE INTO source_tasks(
                source_key, run_id, source_type, source_uri,
                source_payload_json, status, created_at, updated_at
            ) VALUES (?, ?, ?, ?, ?, 'discovered', ?, ?)
            """,
            (
                key, RUN_ID, source_type, source_uri,
                json.dumps(payload, ensure_ascii=False) if payload is not None else None,
                now, now,
            ),
        )
    return key

def _dedupe_urls(urls: Iterable[str]) -> List[str]:
    result = []
    seen = set()

    for url in urls:
        url = str(url or "").strip().rstrip(",;")
        if url and url not in seen:
            seen.add(url)
            result.append(url)

    return result

def _extract_urls_from_values(values: Iterable[Any]) -> List[str]:
    urls: List[str] = []

    for value in values:
        text = str(value or "").strip()
        if not text:
            continue

        urls.extend(re.findall(r"https?://[^\s<>\"']+", text))

    return _dedupe_urls(urls)

def read_url_file(path: Path) -> List[str]:
    suffix = path.suffix.lower()

    if suffix == ".txt":
        values: List[str] = []
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                values.append(line)

        return _extract_urls_from_values(values)

    if suffix in {".xlsx", ".xlsm"}:
        workbook = openpyxl.load_workbook(
            path,
            read_only=False,
            data_only=False,
        )
        values: List[Any] = []

        try:
            for worksheet in workbook.worksheets:
                for row in worksheet.iter_rows():
                    for cell in row:
                        if cell.value is not None:
                            values.append(cell.value)

                        if cell.hyperlink and cell.hyperlink.target:
                            values.append(cell.hyperlink.target)
        finally:
            workbook.close()

        return _extract_urls_from_values(values)

    if suffix == ".xls":
        workbook = xlrd.open_workbook(str(path), on_demand=True)
        values: List[Any] = []

        try:
            for worksheet in workbook.sheets():
                for row_index in range(worksheet.nrows):
                    values.extend(worksheet.row_values(row_index))
        finally:
            workbook.release_resources()

        return _extract_urls_from_values(values)

    raise ValueError(
        f"Unsupported URL file type: {path.suffix}. "
        "Expected .txt, .xlsx, .xlsm, or .xls."
    )

def discover_sources():
    discovered = 0

    if URL_FILE:
        for url in read_url_file(URL_FILE):
            register_source_task("web", url)
            discovered += 1
            if DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT:
                break

    if not (DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT):
        pdf_roots: List[Path] = []

        if PDF_ZIP_FILE:
            extracted_dir = STATE_DIR / "_pdf_zip_extracted"
            extracted_dir.mkdir(parents=True, exist_ok=True)
            marker = extracted_dir / ".extracted_complete"
            if not marker.exists():
                with zipfile.ZipFile(PDF_ZIP_FILE, "r") as zf:
                    zf.extractall(extracted_dir)
                marker.write_text(utc_now(), encoding="utf-8")
            pdf_roots.append(extracted_dir)

        if PDF_DIR:
            pdf_roots.append(PDF_DIR)

        seen_pdf_paths = set()
        for root in pdf_roots:
            for path in sorted(
                p for p in root.rglob("*")
                if p.is_file() and p.suffix.lower() == ".pdf"
            ):
                resolved = str(path.resolve())
                if resolved in seen_pdf_paths:
                    continue
                seen_pdf_paths.add(resolved)
                register_source_task("pdf", resolved)
                discovered += 1
                if DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT:
                    break
            if DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT:
                break

    if not (DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT):
        for cfg in CSV_INPUTS:
            csv_path = Path(cfg["path"]).resolve()
            with open(csv_path, newline="", encoding="utf-8", errors="ignore") as f:
                reader = csv.DictReader(f)
                for row_number, row in enumerate(reader, start=1):
                    source_uri = f"{csv_path}#row={row_number}"
                    register_source_task(
                        "csv",
                        source_uri,
                        payload={
                            "row": row,
                            "row_number": row_number,
                            "csv_path": str(csv_path),
                            "config": {
                                k: v
                                for k, v in cfg.items()
                                if k != "path"
                            },
                        },
                    )
                    discovered += 1
                    if DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT:
                        break
            if DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT:
                break

    with get_db() as conn:
        total = conn.execute(
            "SELECT COUNT(*) FROM source_tasks WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()[0]
    print(f"Source discovery complete. Registered for this run: {total}")

def _find_duplicate_by_raw_hash(raw_hash: str) -> Optional[str]:
    if not raw_hash:
        return None
    with get_db() as conn:
        row = conn.execute(
            "SELECT document_id FROM documents WHERE raw_hash=? LIMIT 1",
            (raw_hash,),
        ).fetchone()
    return row["document_id"] if row else None

def _find_existing_document_by_source_uri(source_type: str, source_uri: str) -> Optional[str]:
    """Reuse a prior canonical document when the exact same source URI was already resolved."""
    with get_db() as conn:
        row = conn.execute(
            """
            SELECT document_id
            FROM source_tasks
            WHERE source_type=? AND source_uri=?
              AND document_id IS NOT NULL
              AND status IN ('extracted','duplicate_skipped')
              AND run_id != ?
            ORDER BY updated_at DESC
            LIMIT 1
            """,
            (source_type, source_uri, RUN_ID),
        ).fetchone()
    return row["document_id"] if row else None

def _register_duplicate(source_row: sqlite3.Row, existing_document_id: str):
    now = utc_now()
    with get_db() as conn:
        conn.execute(
            """
            INSERT OR IGNORE INTO run_documents(
                run_id, document_id, source_type, source_uri,
                duplicate, document_status, created_at, updated_at
            ) VALUES (?, ?, ?, ?, 1, 'duplicate_skipped', ?, ?)
            """,
            (
                RUN_ID, existing_document_id, source_row["source_type"],
                source_row["source_uri"], now, now,
            ),
        )
        conn.execute(
            """
            UPDATE source_tasks
            SET status='duplicate_skipped', document_id=?, duplicate=1,
                source_payload_json=NULL, last_error=NULL, updated_at=?
            WHERE source_key=?
            """,
            (existing_document_id, now, source_row["source_key"]),
        )

def _persist_new_canonical_document(
    source_row: sqlite3.Row,
    extraction: Dict[str, Any],
) -> str:
    text = extraction["text"]
    doc_hash = content_hash(text)
    document_id = doc_hash

    # Extraction itself is parallel, but the global content-hash claim and canonical write
    # are serialized inside this notebook process so two equal sources cannot race each other.
    with CANONICAL_PERSIST_LOCK:
        with get_db() as conn:
            duplicate = conn.execute(
                "SELECT document_id FROM documents WHERE content_hash=?",
                (doc_hash,),
            ).fetchone()

        if duplicate:
            _register_duplicate(source_row, duplicate["document_id"])
            return duplicate["document_id"]

        text_path, metadata_path = canonical_paths(document_id)
        write_zstd_text(text_path, text)

        county = (extraction.get("source_metadata") or {}).get("county", "")
        location = STATE_NAME if not county else f"{STATE_NAME}, {county}"

        metadata = {
            "schema_version": "1.0",
            "document_id": document_id,
            "content_hash": doc_hash,
            "raw_hash": extraction.get("raw_hash", ""),
            "source_type": source_row["source_type"],
            "source_uri": source_row["source_uri"],
            "source_name": Path(source_row["source_uri"].split("#", 1)[0]).name
                if source_row["source_type"] in {"pdf", "csv"} else source_row["source_uri"],
            "run_discovered": RUN_ID,
            "state_discovered": STATE_NAME,
            "title": extraction["title"],
            "language": "en",
            "location": location,
            "month_year": extraction.get("month_year", ""),
            "url": extraction.get("url", ""),
            "canonical_text_path": str(text_path),
            "canonical_text_chars": len(text),
            "canonical_text_bytes": len(text.encode("utf-8")),
            "pages": extraction.get("pages", [{"page": -1, "start": 0, "end": len(text)}]),
            "extraction": {
                "extractor_version": EXTRACTOR_VERSION,
                "extracted_at": utc_now(),
            },
            "source_metadata": extraction.get("source_metadata", {}),
        }
        write_json_atomic(metadata_path, metadata)

        now = utc_now()
        with get_db() as conn:
            conn.execute(
                """
                INSERT INTO documents(
                    document_id, content_hash, raw_hash, source_type,
                    canonical_text_path, canonical_metadata_path,
                    canonical_text_chars, canonical_text_bytes,
                    language, extractor_version, extraction_status,
                    first_seen_run_id, first_seen_state, created_at, updated_at
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 'extracted', ?, ?, ?, ?)
                """,
                (
                    document_id, doc_hash, extraction.get("raw_hash", ""),
                    source_row["source_type"], str(text_path), str(metadata_path),
                    len(text), len(text.encode("utf-8")), "en",
                    EXTRACTOR_VERSION, RUN_ID, STATE_NAME, now, now,
                ),
            )
            conn.execute(
                """
                INSERT INTO run_documents(
                    run_id, document_id, source_type, source_uri,
                    duplicate, document_status, created_at, updated_at
                ) VALUES (?, ?, ?, ?, 0, 'extracted', ?, ?)
                """,
                (
                    RUN_ID, document_id, source_row["source_type"],
                    source_row["source_uri"], now, now,
                ),
            )
            conn.execute(
                """
                UPDATE source_tasks
                SET status='extracted', document_id=?, source_payload_json=NULL,
                    last_error=NULL, updated_at=?
                WHERE source_key=?
                """,
                (document_id, now, source_row["source_key"]),
            )

    return document_id

def extract_source_task(source_row: sqlite3.Row) -> str:
    source_type = source_row["source_type"]
    source_uri = source_row["source_uri"]
    attempt_number = int(source_row["attempt_count"]) + 1
    started = utc_now()

    try:
        # If the exact same URI/path/CSV-row identifier was already resolved in a prior state,
        # reuse the canonical document immediately. Content-hash dedupe remains the final authority.
        prior_document_id = _find_existing_document_by_source_uri(source_type, source_uri)
        if prior_document_id:
            _register_duplicate(source_row, prior_document_id)
            record_attempt(
                "source", source_row["source_key"], "extract",
                attempt_number, "duplicate", started,
            )
            return "duplicate_skipped"

        # Fast raw-file dedupe for PDFs, including renamed/moved copies.
        if source_type == "pdf":
            path = Path(source_uri)
            raw_hash = bytes_sha256(path)
            existing = _find_duplicate_by_raw_hash(raw_hash)
            if existing:
                _register_duplicate(source_row, existing)
                record_attempt("source", source_row["source_key"], "extract", attempt_number, "duplicate", started)
                return "duplicate_skipped"
            extraction = extract_pdf_document(path)

        elif source_type == "web":
            extraction = extract_url_document(source_uri)

        elif source_type == "csv":
            payload = json.loads(source_row["source_payload_json"] or "{}")
            row = payload["row"]
            csv_path = Path(payload["csv_path"])
            row_number = int(payload["row_number"])
            cfg = payload.get("config") or {}

            # Fast raw-record dedupe.
            canonical_raw = json.dumps(
                {str(k): "" if v is None else str(v) for k, v in row.items()},
                sort_keys=True,
                ensure_ascii=False,
                separators=(",", ":"),
            )
            raw_hash = stable_hash(canonical_raw)
            existing = _find_duplicate_by_raw_hash(raw_hash)
            if existing:
                _register_duplicate(source_row, existing)
                record_attempt("source", source_row["source_key"], "extract", attempt_number, "duplicate", started)
                return "duplicate_skipped"

            extraction = extract_csv_row_document(row, row_number, csv_path, cfg)

        else:
            raise ValueError(f"Unsupported source_type: {source_type}")

        _persist_new_canonical_document(source_row, extraction)
        record_attempt("source", source_row["source_key"], "extract", attempt_number, "succeeded", started)
        return "succeeded"

    except Exception as exc:
        with get_db() as conn:
            conn.execute(
                """
                UPDATE source_tasks
                SET status='failed', attempt_count=attempt_count+1,
                    last_error=?, updated_at=?
                WHERE source_key=?
                """,
                (traceback.format_exc()[-4000:], utc_now(), source_row["source_key"]),
            )
        record_attempt("source", source_row["source_key"], "extract", attempt_number, "failed", started, exc)
        return "failed"

def extraction_pass(statuses: Sequence[str] = ("discovered", "failed")):
    placeholders = ",".join("?" for _ in statuses)
    with get_db() as conn:
        rows = conn.execute(
            f"""
            SELECT * FROM source_tasks
            WHERE run_id=? AND status IN ({placeholders})
            ORDER BY created_at, source_key
            """,
            (RUN_ID, *statuses),
        ).fetchall()

    if not rows:
        print("No source tasks for this extraction pass.")
        return

    # URL/PDF extraction benefits from threads. CSV row conversion is cheap but safe in the same pool.
    with ThreadPoolExecutor(max_workers=EXTRACTION_WORKERS) as executor:
        futures = {executor.submit(extract_source_task, row): row["source_key"] for row in rows}
        counts = Counter()
        for future in tqdm(as_completed(futures), total=len(futures), desc="Extract + canonicalize"):
            try:
                counts[future.result()] += 1
            except Exception:
                counts["executor_error"] += 1
    print("Extraction pass:", dict(counts))

def retry_failed_extraction():
    for retry_index in range(MAX_STAGE_RETRIES):
        with get_db() as conn:
            failed = conn.execute(
                """
                SELECT COUNT(*) FROM source_tasks
                WHERE run_id=? AND status='failed'
                """,
                (RUN_ID,),
            ).fetchone()[0]
        if failed == 0:
            break
        print(f"Extraction retry {retry_index + 1}/{MAX_STAGE_RETRIES}: {failed} sources")
        extraction_pass(("failed",))

    with get_db() as conn:
        conn.execute(
            """
            UPDATE source_tasks
            SET status='permanently_failed', updated_at=?
            WHERE run_id=? AND status='failed'
            """,
            (utc_now(), RUN_ID),
        )

print("Discovery/extraction orchestration ready.")

## 10. Qualification model + original classifier contract

In [ ]:
classifier_tokenizer = None
classifier_model = None

def load_classifier_model():
    global classifier_tokenizer, classifier_model, HF_TOKEN

    if classifier_model is not None:
        return

    if not HF_TOKEN:
        HF_TOKEN = input("Paste your Hugging Face token: ").strip()

    if HF_TOKEN:
        login(token=HF_TOKEN)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    classifier_tokenizer = AutoTokenizer.from_pretrained(
        CLASSIFIER_MODEL_ID,
        token=HF_TOKEN or None,
    )
    classifier_model = AutoModelForCausalLM.from_pretrained(
        CLASSIFIER_MODEL_ID,
        token=HF_TOKEN or None,
        quantization_config=bnb_config,
        device_map="auto",
    )

    if classifier_tokenizer.pad_token is None:
        classifier_tokenizer.pad_token = classifier_tokenizer.eos_token

    print("Loaded classifier:", CLASSIFIER_MODEL_ID)

def unload_classifier_model():
    global classifier_tokenizer, classifier_model
    classifier_model = None
    classifier_tokenizer = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Classifier unloaded.")

In [ ]:
SYSTEM_PROMPT = """
You are an information extraction system.

Your task is to analyze agricultural content and return ONLY structured JSON output.

STRICT RULES:
- Output ONLY valid JSON. No explanations, no markdown, no extra text.
- Follow the schema exactly.
- Do NOT hallucinate information.
- Only extract information explicitly present in the text.
- If unsure, return empty lists.
- Do not include duplicate values.

CLASSIFICATION RULES:
You must classify the document chunk into ONE of:
- "crops"
- "pest"
- "disease"
- "management"
- "multi"
- "msc"

Definitions:
- "crops": general crop-related information such as growth, varieties, cultivation, planting, harvest, crop descriptions, or crop production.
- "pest": insects, mites, nematodes, animals, weeds, or other pests damaging crops.
- "disease": fungal, bacterial, viral, oomycete, nematode-caused, or physiological plant diseases/disorders.
- "management": treatment, prevention, farming practices, pesticide/herbicide/fungicide use, scouting, control strategies, irrigation, fertility, planting, harvest, storage, or integrated pest management.
- "multi": use only if multiple categories are clearly present.
- "msc": use if the content is not related to agriculture, crops, pests, diseases, or crop management.

TAG RULES:
- Use "multi" only if more than one category is strongly present.
- If "multi", provide the active categories in "subtags".
- If not "multi", "subtags" MUST be [].
- "subtags" may contain only: "crops", "pest", "disease", "management".
- Never put "multi" or "msc" inside subtags.

CROP RULES:
- You will be given a crop list.
- Match crops case-insensitively.
- Return crop names in lowercase.
- Return only crops that appear in the content.
- If a crop appears in the content but is not in the crop list, still include it.
- Do not invent crop names.

ENTITY EXTRACTION RULES:
- Extract only specific diseases, pests, and management practices explicitly present in the content.
- Normalize all extracted entities to lowercase.
- Keep entities short and specific.
- Do NOT include generic words such as "disease", "pest", "issue", "problem", "management", "control", "crop", or "plant".
- If no specific entity is present, return an empty list for that field.

RELATIONSHIP RULE:
- Preserve relationships between crops and entities.
- Each crop must only contain diseases, pests, and management practices relevant to that crop.
- Do NOT assign every entity to every crop unless the content clearly says the entity applies to all those crops.
- If the content explicitly discusses multiple crops separately, keep their entities separate.

STRICT CONSISTENCY:
- If tag = "disease", only fill "disease"; "pests" and "management" must be empty.
- If tag = "pest", only fill "pests"; "disease" and "management" must be empty.
- If tag = "management", only fill "management"; "disease" and "pests" must be empty.
- If tag = "crops", crop_entities may contain crops with empty entity lists.
- If tag = "multi", fill only fields corresponding to subtags.
- If tag = "msc", crop_entities must be {}.

OUTPUT SCHEMA:
{
  "tag": "string",
  "subtags": ["string"],
  "crop_entities": {
    "crop_name": {
      "disease": ["string"],
      "pests": ["string"],
      "management": ["string"]
    }
  }
}
"""

def build_user_prompt(crop_list: List[str], content: str) -> str:
    crop_text = json.dumps(crop_list, ensure_ascii=False)
    return f"""CROP LIST:
{crop_text}

CONTENT:
{content}
"""

In [ ]:
def build_user_prompt(crop_list: List[str], content: str) -> str:
    crop_text = json.dumps(crop_list, ensure_ascii=False)
    return f"""CROP LIST:
{crop_text}

CONTENT:
{content}
"""

def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None

    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    try:
        value = json.loads(text)
        return value if isinstance(value, dict) else None
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end > start:
        try:
            value = json.loads(text[start:end + 1])
            return value if isinstance(value, dict) else None
        except Exception:
            return None
    return None

def run_llm_once(content: str, crop_list: List[str], max_new_tokens: int = 900) -> str:
    if classifier_model is None:
        load_classifier_model()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(crop_list, content)},
    ]
    prompt = classifier_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = classifier_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=12000,
    ).to(classifier_model.device)

    with torch.no_grad():
        output_ids = classifier_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=classifier_tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return classifier_tokenizer.decode(generated, skip_special_tokens=True).strip()

def fallback_msc() -> Dict[str, Any]:
    return {"tag": "msc", "subtags": [], "crop_entities": {}}

def crops_found_by_string_match(text: str, crop_list: List[str]) -> Set[str]:
    text_norm = normalize_name(text)
    found = set()
    for crop in crop_list:
        pattern = r"(?<![a-z0-9])" + re.escape(crop) + r"(?![a-z0-9])"
        if re.search(pattern, text_norm):
            found.add(crop)
    return found

def crop_appears_in_text(crop: str, text: str) -> bool:
    crop_norm = normalize_name(crop)
    text_norm = normalize_name(text)
    pattern = r"(?<![a-z0-9])" + re.escape(crop_norm) + r"(?![a-z0-9])"
    return re.search(pattern, text_norm) is not None

def validate_and_clean_llm_output(raw: Any, text: str, crop_list: List[str]) -> Dict[str, Any]:
    if not isinstance(raw, dict):
        raise ValueError("LLM output is not a JSON object")

    tag = normalize_name(raw.get("tag", ""))
    if tag not in ALLOWED_TAGS:
        raise ValueError(f"Invalid tag: {tag!r}")

    subtags = raw.get("subtags", [])
    if not isinstance(subtags, list):
        raise ValueError("subtags must be a list")
    subtags = sorted({
        normalize_name(s)
        for s in subtags
        if normalize_name(s) in {"crops", "pest", "disease", "management"}
    })

    if tag != "multi":
        subtags = []

    crop_entities = raw.get("crop_entities", {})
    if tag == "msc":
        return fallback_msc()
    if not isinstance(crop_entities, dict):
        raise ValueError("crop_entities must be an object")

    valid_known_crops_in_text = crops_found_by_string_match(text, crop_list)
    cleaned_crop_entities = {}

    for crop_name, fields in crop_entities.items():
        crop_norm = normalize_name(crop_name)
        if not crop_norm:
            continue

        if crop_norm in crop_list:
            if crop_norm not in valid_known_crops_in_text:
                continue
        elif not crop_appears_in_text(crop_norm, text):
            continue

        if not isinstance(fields, dict):
            fields = {}

        cleaned_fields = {"disease": [], "pests": [], "management": []}
        for field in ENTITY_FIELDS:
            values = fields.get(field, [])
            if not isinstance(values, list):
                values = []
            cleaned_fields[field] = sorted({
                ent
                for ent in (normalize_entity(v) for v in values)
                if ent
            })

        cleaned_crop_entities[crop_norm] = cleaned_fields

    if tag == "multi":
        allowed_fields = set()
        if "disease" in subtags:
            allowed_fields.add("disease")
        if "pest" in subtags:
            allowed_fields.add("pests")
        if "management" in subtags:
            allowed_fields.add("management")
    else:
        allowed_fields = {
            "crops": set(),
            "disease": {"disease"},
            "pest": {"pests"},
            "management": {"management"},
        }.get(tag, set())

    for fields in cleaned_crop_entities.values():
        for field in ENTITY_FIELDS:
            if field not in allowed_fields:
                fields[field] = []

    if tag == "multi":
        inferred = set(subtags)
        for fields in cleaned_crop_entities.values():
            if fields["disease"]:
                inferred.add("disease")
            if fields["pests"]:
                inferred.add("pest")
            if fields["management"]:
                inferred.add("management")
        subtags = sorted(inferred)

        if len(subtags) < 2:
            for candidate in ["disease", "pest", "management", "crops"]:
                if candidate in subtags:
                    tag, subtags = candidate, []
                    break
            else:
                return fallback_msc()

    return {
        "tag": tag,
        "subtags": subtags,
        "crop_entities": cleaned_crop_entities,
    }

def meaningful_categories_from_output(output: Dict[str, Any]) -> Set[str]:
    tag = output.get("tag", "msc")
    if tag == "multi":
        return set(output.get("subtags", [])) - {"msc", "multi"}
    if tag in {"crops", "pest", "disease", "management"}:
        return {tag}
    return set()

def merge_validated_outputs(outputs: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    categories = set()
    merged_entities = defaultdict(
        lambda: {"disease": set(), "pests": set(), "management": set()}
    )

    for validated in outputs:
        categories.update(meaningful_categories_from_output(validated))
        for crop, fields in (validated.get("crop_entities") or {}).items():
            crop_norm = normalize_name(crop)
            if not crop_norm:
                continue
            for field in ENTITY_FIELDS:
                for ent in fields.get(field, []) or []:
                    norm_ent = normalize_entity(ent)
                    if norm_ent:
                        merged_entities[crop_norm][field].add(norm_ent)

    if not categories:
        return fallback_msc()

    final_tag = next(iter(categories)) if len(categories) == 1 else "multi"
    subtags = [] if final_tag != "multi" else sorted(categories)

    crop_entities = {
        crop: {
            "disease": sorted(fields["disease"]),
            "pests": sorted(fields["pests"]),
            "management": sorted(fields["management"]),
        }
        for crop, fields in merged_entities.items()
    }

    allowed = set()
    if final_tag == "disease":
        allowed = {"disease"}
    elif final_tag == "pest":
        allowed = {"pests"}
    elif final_tag == "management":
        allowed = {"management"}
    elif final_tag == "multi":
        if "disease" in subtags:
            allowed.add("disease")
        if "pest" in subtags:
            allowed.add("pests")
        if "management" in subtags:
            allowed.add("management")

    for fields in crop_entities.values():
        for field in ENTITY_FIELDS:
            if field not in allowed:
                fields[field] = []

    return {
        "tag": final_tag,
        "subtags": subtags,
        "crop_entities": crop_entities,
    }

def total_entity_count(doc_output: Dict[str, Any]) -> int:
    return sum(
        len(fields.get(field, []))
        for fields in doc_output.get("crop_entities", {}).values()
        for field in ENTITY_FIELDS
    )

def has_crop_and_entity(doc_output: Dict[str, Any]) -> bool:
    return bool(doc_output.get("crop_entities")) and total_entity_count(doc_output) >= 1

print("Qualification contract ready.")

## 11. Qualification chunk persistence, cache, retries, and document decision

The initial pass processes all documents. Failed qualification chunks are retried **after** the pass, scoped to this `RUN_ID`. Remaining failures become `permanently_failed`.

A document can still be decided from successful chunks if some sibling chunks permanently fail; those failures remain auditable in the manifest.

In [ ]:
def qualification_chunks_for_text(text: str) -> List[str]:
    text = text.strip()
    if not text:
        return []

    chunks = []
    start = 0
    while start < len(text) and len(chunks) < MAX_QUALIFICATION_CHUNKS:
        end = min(start + QUALIFICATION_CHUNK_CHARS, len(text))
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = max(0, end - QUALIFICATION_OVERLAP_CHARS)
    return chunks

def ensure_qualification_chunk_rows(document_id: str):
    with get_db() as conn:
        doc = conn.execute(
            "SELECT canonical_text_path FROM documents WHERE document_id=?",
            (document_id,),
        ).fetchone()
    text = read_zstd_text(Path(doc["canonical_text_path"]))
    chunks = qualification_chunks_for_text(text)

    now = utc_now()
    with get_db() as conn:
        for i, chunk in enumerate(chunks):
            qid = stable_hash(f"{RUN_ID}|{document_id}|q|{i}|{CLASSIFIER_VERSION}")
            conn.execute(
                """
                INSERT OR IGNORE INTO qualification_chunks(
                    qualification_chunk_id, run_id, document_id,
                    chunk_index, chunk_hash, status,
                    created_at, updated_at
                ) VALUES (?, ?, ?, ?, ?, 'pending', ?, ?)
                """,
                (qid, RUN_ID, document_id, i, content_hash(chunk), now, now),
            )

def qualification_cache_key(chunk: str) -> Tuple[str, str, str]:
    crop_hash = stable_hash(json.dumps(CROP_LIST, sort_keys=True))
    chunk_content_hash = content_hash(chunk)
    payload = {
        "model_id": CLASSIFIER_MODEL_ID,
        "classifier_version": CLASSIFIER_VERSION,
        "state": STATE_KEY,
        "crop_hash": crop_hash,
        "content_hash": chunk_content_hash,
    }
    return stable_hash(json.dumps(payload, sort_keys=True)), crop_hash, chunk_content_hash

def classify_chunk_one_attempt(chunk: str) -> Tuple[Dict[str, Any], str]:
    cache_key, crop_hash, chunk_content_hash = qualification_cache_key(chunk)

    with get_db() as conn:
        cached = conn.execute(
            "SELECT * FROM classification_cache WHERE cache_key=?",
            (cache_key,),
        ).fetchone()

    if cached:
        return json.loads(cached["validated_output_json"]), cached["raw_output"] or ""

    raw_text = run_llm_once(chunk, CROP_LIST)
    parsed = extract_json_object(raw_text)
    if parsed is None:
        raise ValueError("Classifier did not return parseable JSON")

    cleaned = validate_and_clean_llm_output(parsed, chunk, CROP_LIST)

    with get_db() as conn:
        conn.execute(
            """
            INSERT OR REPLACE INTO classification_cache(
                cache_key, model_id, classifier_version, state_key,
                crop_hash, content_hash, validated_output_json,
                raw_output, created_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                cache_key, CLASSIFIER_MODEL_ID, CLASSIFIER_VERSION,
                STATE_KEY, crop_hash, chunk_content_hash,
                json.dumps(cleaned, ensure_ascii=False), raw_text, utc_now(),
            ),
        )

    return cleaned, raw_text

def _load_document_text(document_id: str) -> str:
    with get_db() as conn:
        row = conn.execute(
            "SELECT canonical_text_path FROM documents WHERE document_id=?",
            (document_id,),
        ).fetchone()
    if row is None:
        raise KeyError(document_id)
    return read_zstd_text(Path(row["canonical_text_path"]))

def _successful_doc_outputs(document_id: str) -> List[Dict[str, Any]]:
    with get_db() as conn:
        rows = conn.execute(
            """
            SELECT classification_result_json
            FROM qualification_chunks
            WHERE run_id=? AND document_id=? AND status='succeeded'
            ORDER BY chunk_index
            """,
            (RUN_ID, document_id),
        ).fetchall()
    return [
        json.loads(row["classification_result_json"])
        for row in rows
        if row["classification_result_json"]
    ]

def process_qualification_chunk(row: sqlite3.Row) -> str:
    document_id = row["document_id"]
    text = _load_document_text(document_id)
    chunks = qualification_chunks_for_text(text)
    idx = int(row["chunk_index"])

    if idx >= len(chunks):
        raise IndexError(f"Qualification chunk index {idx} no longer exists")

    chunk = chunks[idx]
    attempt_number = int(row["attempt_count"]) + 1
    started = utc_now()

    try:
        cleaned, raw_text = classify_chunk_one_attempt(chunk)

        with get_db() as conn:
            conn.execute(
                """
                UPDATE qualification_chunks
                SET status='succeeded',
                    classification_result_json=?,
                    attempt_count=attempt_count+1,
                    last_error=NULL,
                    updated_at=?
                WHERE qualification_chunk_id=?
                """,
                (
                    json.dumps(cleaned, ensure_ascii=False),
                    utc_now(), row["qualification_chunk_id"],
                ),
            )
        record_attempt(
            "qualification_chunk", row["qualification_chunk_id"],
            "classify", attempt_number, "succeeded", started,
        )
        return "succeeded"

    except Exception as exc:
        with get_db() as conn:
            conn.execute(
                """
                UPDATE qualification_chunks
                SET status='failed',
                    attempt_count=attempt_count+1,
                    last_error=?,
                    updated_at=?
                WHERE qualification_chunk_id=?
                """,
                (
                    traceback.format_exc()[-4000:],
                    utc_now(), row["qualification_chunk_id"],
                ),
            )
        record_attempt(
            "qualification_chunk", row["qualification_chunk_id"],
            "classify", attempt_number, "failed", started, exc,
        )
        return "failed"

def maybe_apply_early_stop(document_id: str):
    outputs = _successful_doc_outputs(document_id)
    merged = merge_validated_outputs(outputs)

    if (
        total_entity_count(merged) >= EARLY_STOP_TOTAL_ENTITIES
        or has_crop_and_entity(merged)
    ):
        with get_db() as conn:
            conn.execute(
                """
                UPDATE qualification_chunks
                SET status='skipped_early_stop', updated_at=?
                WHERE run_id=? AND document_id=? AND status='pending'
                """,
                (utc_now(), RUN_ID, document_id),
            )
        return True
    return False

def qualification_initial_pass():
    with get_db() as conn:
        docs = conn.execute(
            """
            SELECT document_id
            FROM run_documents
            WHERE run_id=? AND duplicate=0
              AND document_status IN ('extracted', 'qualifying')
            ORDER BY created_at, document_id
            """,
            (RUN_ID,),
        ).fetchall()

    if not docs:
        print("No documents awaiting qualification.")
        return

    load_classifier_model()

    for doc_row in tqdm(docs, desc="Qualifying documents"):
        document_id = doc_row["document_id"]
        ensure_qualification_chunk_rows(document_id)

        with get_db() as conn:
            conn.execute(
                """
                UPDATE run_documents
                SET document_status='qualifying',
                    qualification_status='processing',
                    classifier_version=?,
                    updated_at=?
                WHERE run_id=? AND document_id=?
                """,
                (CLASSIFIER_VERSION, utc_now(), RUN_ID, document_id),
            )
            chunks = conn.execute(
                """
                SELECT * FROM qualification_chunks
                WHERE run_id=? AND document_id=? AND status='pending'
                ORDER BY chunk_index
                """,
                (RUN_ID, document_id),
            ).fetchall()

        for row in chunks:
            process_qualification_chunk(row)
            maybe_apply_early_stop(document_id)

            with get_db() as conn:
                still_pending = conn.execute(
                    """
                    SELECT COUNT(*) FROM qualification_chunks
                    WHERE run_id=? AND document_id=? AND status='pending'
                    """,
                    (RUN_ID, document_id),
                ).fetchone()[0]
            if still_pending == 0:
                break

def retry_failed_qualification():
    for retry_idx in range(MAX_STAGE_RETRIES):
        with get_db() as conn:
            failed_rows = conn.execute(
                """
                SELECT * FROM qualification_chunks
                WHERE run_id=? AND status='failed'
                ORDER BY document_id, chunk_index
                """,
                (RUN_ID,),
            ).fetchall()

        if not failed_rows:
            break

        print(
            f"Qualification retry {retry_idx + 1}/{MAX_STAGE_RETRIES}: "
            f"{len(failed_rows)} failed chunks"
        )
        load_classifier_model()

        for row in tqdm(failed_rows, desc=f"Qualification retry {retry_idx + 1}"):
            process_qualification_chunk(row)

    with get_db() as conn:
        conn.execute(
            """
            UPDATE qualification_chunks
            SET status='permanently_failed', updated_at=?
            WHERE run_id=? AND status='failed'
            """,
            (utc_now(), RUN_ID),
        )

def finalize_qualification_decisions():
    with get_db() as conn:
        docs = conn.execute(
            """
            SELECT document_id
            FROM run_documents
            WHERE run_id=? AND duplicate=0
              AND document_status NOT IN (
                  'rejected', 'accepted', 'rag_preparing',
                  'indexing', 'indexed', 'permanently_failed'
              )
            """,
            (RUN_ID,),
        ).fetchall()

    for row in docs:
        document_id = row["document_id"]

        with get_db() as conn:
            nonterminal = conn.execute(
                """
                SELECT COUNT(*) FROM qualification_chunks
                WHERE run_id=? AND document_id=?
                  AND status IN ('pending','processing','failed')
                """,
                (RUN_ID, document_id),
            ).fetchone()[0]
            succeeded = conn.execute(
                """
                SELECT COUNT(*) FROM qualification_chunks
                WHERE run_id=? AND document_id=? AND status='succeeded'
                """,
                (RUN_ID, document_id),
            ).fetchone()[0]

        if nonterminal:
            continue

        if succeeded == 0:
            with get_db() as conn:
                conn.execute(
                    """
                    UPDATE run_documents
                    SET document_status='permanently_failed',
                        qualification_status='permanently_failed',
                        accepted=0,
                        qualification_reason='No qualification chunk succeeded',
                        updated_at=?
                    WHERE run_id=? AND document_id=?
                    """,
                    (utc_now(), RUN_ID, document_id),
                )
            continue

        merged = merge_validated_outputs(_successful_doc_outputs(document_id))
        accepted = merged["tag"] != "msc"

        with get_db() as conn:
            conn.execute(
                """
                UPDATE run_documents
                SET document_status=?,
                    qualification_status='succeeded',
                    accepted=?,
                    qualification_tag=?,
                    qualification_subtags_json=?,
                    qualification_entities_json=?,
                    qualification_reason=?,
                    classifier_version=?,
                    updated_at=?
                WHERE run_id=? AND document_id=?
                """,
                (
                    "accepted" if accepted else "rejected",
                    int(accepted),
                    merged["tag"],
                    json.dumps(merged["subtags"], ensure_ascii=False),
                    json.dumps(merged["crop_entities"], ensure_ascii=False),
                    "tag != msc" if accepted else "tag == msc",
                    CLASSIFIER_VERSION,
                    utc_now(), RUN_ID, document_id,
                ),
            )

    print("Qualification decisions finalized.")

print("Qualification persistence/retry pipeline ready.")

## 12. Runtime-compatible RAG chunking

Contract:

- Tokenizer: `BAAI/bge-base-en-v1.5`
- Max RAG chunk: 480 tokens
- Overlap: 80
- Hard cap: 512

Source behavior:

- Web ≤512 tokens → one chunk; otherwise 480/80.
- PDF → each page chunked independently at 480/80.
- CSV row ≤512 → one chunk; otherwise 480/80.

In [ ]:
rag_tokenizer = None

def load_rag_tokenizer():
    global rag_tokenizer
    if rag_tokenizer is None:
        rag_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
    return rag_tokenizer

def token_chunks(text: str, max_tokens: int = RAG_CHUNK_SIZE, overlap: int = RAG_CHUNK_OVERLAP) -> List[str]:
    tokenizer = load_rag_tokenizer()
    max_tokens = min(int(max_tokens), RAG_HARD_CAP)
    tokens = tokenizer.encode(text, add_special_tokens=False, truncation=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end][:RAG_HARD_CAP]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True).strip()
        if chunk_text:
            chunks.append(chunk_text)

        if end == len(tokens):
            break
        start = max(end - overlap, 0)

    return chunks

def canonical_doc_metadata(document_id: str) -> Dict[str, Any]:
    with get_db() as conn:
        row = conn.execute(
            "SELECT canonical_metadata_path FROM documents WHERE document_id=?",
            (document_id,),
        ).fetchone()
    return load_json(Path(row["canonical_metadata_path"]), {})

def reconstruct_page_text(document_id: str, page: int) -> str:
    meta = canonical_doc_metadata(document_id)
    text = _load_document_text(document_id)

    if page == -1:
        return text

    for rec in meta.get("pages", []):
        if int(rec.get("page", -999)) == int(page):
            return text[int(rec["start"]):int(rec["end"])]
    raise KeyError(f"Page {page} not found in canonical metadata for {document_id}")

def build_rag_chunk_specs(document_id: str) -> List[Dict[str, Any]]:
    meta = canonical_doc_metadata(document_id)
    source_type = meta["source_type"]
    specs: List[Dict[str, Any]] = []

    if source_type == "pdf":
        for page_rec in meta.get("pages", []):
            page = int(page_rec["page"])
            page_text = reconstruct_page_text(document_id, page)
            chunks = token_chunks(page_text, RAG_CHUNK_SIZE, RAG_CHUNK_OVERLAP)
            for chunk_index, chunk in enumerate(chunks):
                specs.append({
                    "page": page,
                    "chunk_index": chunk_index,
                    "text": chunk,
                })
    elif source_type in {"web", "csv"}:
        text = reconstruct_page_text(document_id, -1)
        tokenizer = load_rag_tokenizer()
        n_tokens = len(tokenizer.encode(text, add_special_tokens=False, truncation=False))
        chunks = [text] if n_tokens <= RAG_HARD_CAP else token_chunks(
            text, RAG_CHUNK_SIZE, RAG_CHUNK_OVERLAP
        )
        for chunk_index, chunk in enumerate(chunks):
            specs.append({
                "page": -1,
                "chunk_index": chunk_index,
                "text": chunk,
            })
    else:
        raise ValueError(f"Unsupported source_type for RAG chunking: {source_type}")

    return specs

print("RAG chunking helpers ready.")

## 13. Metadata enrichment + **hard contract validation**

This is the ingestion gate. Every Qdrant point must contain the full runtime payload contract.

`month_year` must exist but may be `""` when the date cannot be determined. `hardiness_zone=""` is accepted only for AK/HI.

In [ ]:
REQUIRED_QDRANT_FIELDS = [
    "text",
    "chunk_id",
    "source_type",
    "source_id",
    "title",
    "url",
    "page",
    "chunk_index",
    "location",
    "month_year",
    "content_hash",
    "language",
    "hardiness_zone",
]

def build_chunk_id(
    document_id: str,
    page: int,
    chunk_index: int,
) -> str:
    source_id = document_id[:16]
    return f"{source_id}_{CHUNKER_VERSION}_p{page}_c{chunk_index}"

def enrich_chunk_metadata(
    document_id: str,
    page: int,
    chunk_index: int,
    chunk_text: str,
) -> Dict[str, Any]:
    meta = canonical_doc_metadata(document_id)

    title = re.sub(r"\s+", " ", str(meta.get("title") or "")).strip()
    if not title:
        if meta["source_type"] == "pdf":
            title = Path(meta["source_uri"]).stem
        elif meta["source_type"] == "web":
            title = fallback_title_from_url(meta.get("url") or meta["source_uri"])
        else:
            title = f"{STATE_NAME} record {document_id[:12]}"

    location = re.sub(r"\s+", " ", str(meta.get("location") or STATE_NAME)).strip()
    zone = hardiness_zone_for_location(location)

    chunk_id = build_chunk_id(document_id, page, chunk_index)
    chunk_content_hash = content_hash(chunk_text)

    # Runtime ingestion stores a title-prefixed document while hashing the raw chunk.
    stored_text = f"Title: {title}\n\n{chunk_text}"

    return {
        "text": stored_text,
        "chunk_id": chunk_id,
        "source_type": meta["source_type"],
        "source_id": document_id[:16],
        "title": title,
        "url": str(meta.get("url") or ""),
        "page": int(page),
        "chunk_index": int(chunk_index),
        "location": location,
        "month_year": normalize_month_year(meta.get("month_year")) or "",
        "content_hash": chunk_content_hash,
        "language": str(meta.get("language") or "en"),
        "hardiness_zone": zone,
    }

def validate_qdrant_metadata(payload: Dict[str, Any]) -> Dict[str, Any]:
    missing_fields = [field for field in REQUIRED_QDRANT_FIELDS if field not in payload]
    invalid_fields = []

    if not missing_fields:
        nonempty_required = [
            "text", "chunk_id", "source_type", "source_id",
            "title", "location", "content_hash", "language",
        ]
        for field in nonempty_required:
            if not isinstance(payload[field], str) or not payload[field].strip():
                invalid_fields.append(field)

        if not isinstance(payload["url"], str):
            invalid_fields.append("url")

        if not isinstance(payload["page"], int):
            invalid_fields.append("page")
        if not isinstance(payload["chunk_index"], int) or payload["chunk_index"] < 0:
            invalid_fields.append("chunk_index")

        month_year = payload["month_year"]
        if not isinstance(month_year, str):
            invalid_fields.append("month_year")
        elif month_year and not re.fullmatch(r"\d{4}-(0[1-9]|1[0-2])", month_year):
            invalid_fields.append("month_year")

        zone = payload["hardiness_zone"]
        if not isinstance(zone, str):
            invalid_fields.append("hardiness_zone")
        elif not zone.strip() and STATE_CODE not in {"AK", "HI"}:
            invalid_fields.append("hardiness_zone")

    return {
        "valid": not missing_fields and not invalid_fields,
        "missing_fields": sorted(set(missing_fields)),
        "invalid_fields": sorted(set(invalid_fields)),
        "month_year_available": bool(payload.get("month_year")),
        "ak_hi_hardiness_exception": (
            STATE_CODE in {"AK", "HI"} and not str(payload.get("hardiness_zone", "")).strip()
        ),
    }

def prepare_rag_chunks():
    with get_db() as conn:
        docs = conn.execute(
            """
            SELECT document_id
            FROM run_documents
            WHERE run_id=? AND accepted=1
              AND document_status IN ('accepted','rag_preparing','indexing')
            ORDER BY created_at, document_id
            """,
            (RUN_ID,),
        ).fetchall()

    for row in tqdm(docs, desc="Preparing RAG chunks"):
        document_id = row["document_id"]
        specs = build_rag_chunk_specs(document_id)

        with get_db() as conn:
            conn.execute(
                """
                UPDATE run_documents
                SET document_status='rag_preparing', rag_status='preparing', updated_at=?
                WHERE run_id=? AND document_id=?
                """,
                (utc_now(), RUN_ID, document_id),
            )

        for spec in specs:
            page = int(spec["page"])
            chunk_index = int(spec["chunk_index"])
            chunk_text = spec["text"]
            chunk_hash = content_hash(chunk_text)

            chunk_id = build_chunk_id(document_id, page, chunk_index)
            rag_chunk_id = stable_hash(f"{document_id}|{page}|{chunk_index}|{CHUNKER_VERSION}")
            qdrant_point_id = deterministic_point_uuid(chunk_id)

            payload = enrich_chunk_metadata(
                document_id, page, chunk_index, chunk_text
            )
            validation = validate_qdrant_metadata(payload)

            status = "metadata_validated" if validation["valid"] else "failed"
            failure_stage = None if validation["valid"] else "metadata"
            last_error = None if validation["valid"] else json.dumps(validation)

            token_count = len(
                load_rag_tokenizer().encode(
                    chunk_text,
                    add_special_tokens=False,
                    truncation=False,
                )
            )

            inserted = False
            metadata_started = utc_now()
            with get_db() as conn:
                cursor = conn.execute(
                    """
                    INSERT OR IGNORE INTO rag_chunks(
                        rag_chunk_id, run_id, document_id,
                        page, chunk_index, chunk_hash, token_count,
                        chunker_version, metadata_json,
                        status, embedding_status, qdrant_status,
                        metadata_attempt_count, failure_stage, last_error, qdrant_point_id,
                        created_at, updated_at
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 'pending', 'pending', 1, ?, ?, ?, ?, ?)
                    """,
                    (
                        rag_chunk_id, RUN_ID, document_id, page, chunk_index,
                        chunk_hash, token_count, CHUNKER_VERSION,
                        json.dumps(payload, ensure_ascii=False),
                        status, failure_stage, last_error, qdrant_point_id,
                        utc_now(), utc_now(),
                    ),
                )
                inserted = cursor.rowcount == 1

            if inserted:
                record_attempt(
                    "rag_chunk", rag_chunk_id, "metadata", 1,
                    "succeeded" if validation["valid"] else "failed",
                    metadata_started,
                    None if validation["valid"] else ValueError(json.dumps(validation)),
                )

        with get_db() as conn:
            conn.execute(
                """
                UPDATE run_documents
                SET document_status='indexing', rag_status='prepared', updated_at=?
                WHERE run_id=? AND document_id=?
                """,
                (utc_now(), RUN_ID, document_id),
            )

    print("RAG chunk preparation complete.")

print("Metadata contract gate ready.")

## 14. Embedding + Qdrant initialization

The sentence-transformer model matches the runtime embedder. Qdrant is created with cosine distance and payload indexes for `hardiness_zone`, `month_year`, `title`, and `content_hash`.

For cumulative runs after state 1, the continuity guard can restore the previous run's snapshot when the live collection is missing or inconsistent.

In [ ]:
embedding_model = None
qdrant_client = None

def load_embedding_model():
    global embedding_model
    if embedding_model is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        embedding_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
        print(
            "Loaded embedding model:",
            EMBEDDING_MODEL,
            "dimension=",
            embedding_model.get_sentence_embedding_dimension(),
            "device=",
            device,
        )
    return embedding_model

def get_qdrant_client() -> QdrantClient:
    global qdrant_client
    if qdrant_client is None:
        qdrant_client = QdrantClient(
            url=QDRANT_URL,
            api_key=QDRANT_API_KEY,
            timeout=120,
        )
    return qdrant_client

def qdrant_headers() -> Dict[str, str]:
    return {"api-key": QDRANT_API_KEY} if QDRANT_API_KEY else {}

def qdrant_collection_exists() -> bool:
    return get_qdrant_client().collection_exists(QDRANT_COLLECTION)

def qdrant_point_count() -> int:
    if not qdrant_collection_exists():
        return 0
    return int(
        get_qdrant_client().count(
            collection_name=QDRANT_COLLECTION,
            exact=True,
        ).count
    )

def ensure_payload_indexes():
    client = get_qdrant_client()
    for field in ["hardiness_zone", "month_year", "title", "content_hash"]:
        try:
            client.create_payload_index(
                collection_name=QDRANT_COLLECTION,
                field_name=field,
                field_schema=PayloadSchemaType.KEYWORD,
                wait=True,
            )
        except Exception as exc:
            # Existing-index responses differ between Qdrant versions; verify collection later.
            msg = str(exc).lower()
            if "already exists" not in msg and "already" not in msg:
                print(f"Payload index warning for {field}: {exc}")

def _existing_collection_vector_size() -> Optional[int]:
    if not qdrant_collection_exists():
        return None
    info = get_qdrant_client().get_collection(QDRANT_COLLECTION)
    vectors = info.config.params.vectors
    if hasattr(vectors, "size"):
        return int(vectors.size)
    if isinstance(vectors, dict) and vectors:
        first = next(iter(vectors.values()))
        if hasattr(first, "size"):
            return int(first.size)
        if isinstance(first, dict) and "size" in first:
            return int(first["size"])
    return None

def create_collection_if_needed():
    client = get_qdrant_client()
    model = load_embedding_model()
    dimension = int(model.get_sentence_embedding_dimension())

    if client.collection_exists(QDRANT_COLLECTION):
        existing_size = _existing_collection_vector_size()
        if existing_size is not None and existing_size != dimension:
            raise RuntimeError(
                f"Qdrant vector-size mismatch: collection={existing_size}, "
                f"embedding model={dimension}. Use the matching model or a new collection."
            )
        ensure_payload_indexes()
        return

    client.create_collection(
        collection_name=QDRANT_COLLECTION,
        vectors_config=VectorParams(
            size=dimension,
            distance=Distance.COSINE,
        ),
    )
    ensure_payload_indexes()
    print("Created Qdrant collection:", QDRANT_COLLECTION)

def previous_manifest_and_snapshot() -> Tuple[Optional[Dict[str, Any]], Optional[Path]]:
    if not PREVIOUS_RUN_DIR:
        return None, None

    manifest_path = PREVIOUS_RUN_DIR / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Previous manifest missing: {manifest_path}")

    manifest = load_json(manifest_path)
    snapshot_name = manifest.get("snapshot", {}).get("file")
    if not snapshot_name:
        raise ValueError(f"Previous manifest does not name a snapshot: {manifest_path}")

    snapshot_path = PREVIOUS_RUN_DIR / snapshot_name
    if not snapshot_path.exists():
        raise FileNotFoundError(f"Previous snapshot missing: {snapshot_path}")

    return manifest, snapshot_path

def restore_local_snapshot(snapshot_path: Path):
    # Uploading a collection snapshot requires the destination collection to be absent.
    if qdrant_collection_exists():
        requests.delete(
            f"{QDRANT_URL.rstrip('/')}/collections/{QDRANT_COLLECTION}",
            headers=qdrant_headers(),
            timeout=120,
        ).raise_for_status()

    with open(snapshot_path, "rb") as f:
        response = requests.post(
            f"{QDRANT_URL.rstrip('/')}/collections/{QDRANT_COLLECTION}/snapshots/upload",
            params={"priority": "snapshot"},
            headers=qdrant_headers(),
            files={"snapshot": (snapshot_path.name, f, "application/octet-stream")},
            timeout=None,
        )
    response.raise_for_status()
    print("Restored previous cumulative snapshot:", snapshot_path)

def ensure_qdrant_continuity():
    if RUN_SEQUENCE == 1:
        create_collection_if_needed()
        return

    previous_manifest, snapshot_path = previous_manifest_and_snapshot()
    expected_count = None
    if previous_manifest:
        expected_count = int(
            previous_manifest.get("cumulative", {}).get(
                "qdrant_points",
                previous_manifest.get("snapshot", {}).get("qdrant_point_count", 0),
            )
            or 0
        )

    live_exists = qdrant_collection_exists()
    live_count = qdrant_point_count() if live_exists else 0

    if expected_count is not None and live_exists and live_count == expected_count:
        print(f"Qdrant continuity verified: {live_count} prior points.")
        create_collection_if_needed()  # also validates vector dimension + indexes
        return

    if previous_manifest is None:
        if live_exists and live_count > 0:
            print(
                "No PREVIOUS_RUN_DIR configured; using existing live cumulative collection "
                f"with {live_count} points."
            )
            create_collection_if_needed()  # validates vector dimension + indexes
            return
        raise RuntimeError(
            "Cumulative run > 1 requires an existing populated collection or PREVIOUS_RUN_DIR."
        )

    if not AUTO_RESTORE_PREVIOUS_SNAPSHOT:
        raise RuntimeError(
            f"Live collection count ({live_count}) does not match previous manifest "
            f"({expected_count}) and auto-restore is disabled."
        )

    restore_local_snapshot(snapshot_path)
    restored_count = qdrant_point_count()
    if expected_count is not None and restored_count != expected_count:
        raise RuntimeError(
            f"Restored count mismatch: got {restored_count}, expected {expected_count}"
        )
    create_collection_if_needed()  # validates restored collection dimension + indexes

print("Qdrant initialization/continuity helpers ready.")

## 15. Batch embedding + Qdrant upsert + RAG retries

The Qdrant payload is already validated before this stage. The writer does **not** construct metadata.

In [ ]:
def reconstruct_rag_chunk_text(row: sqlite3.Row) -> str:
    specs = build_rag_chunk_specs(row["document_id"])
    for spec in specs:
        if int(spec["page"]) == int(row["page"]) and int(spec["chunk_index"]) == int(row["chunk_index"]):
            return spec["text"]
    raise KeyError(
        f"Cannot reconstruct RAG chunk {row['document_id']} page={row['page']} "
        f"chunk={row['chunk_index']}"
    )

def chunk_hash_already_indexed(chunk_hash: str, exclude_rag_chunk_id: str) -> bool:
    """Global build-level RAG chunk dedupe without one Qdrant HTTP lookup per chunk."""
    with get_db() as conn:
        row = conn.execute(
            """
            SELECT 1
            FROM rag_chunks rc
            JOIN runs r ON r.run_id = rc.run_id
            WHERE r.build_id=?
              AND rc.chunk_hash=?
              AND rc.rag_chunk_id != ?
              AND rc.status IN ('indexed','duplicate_skipped')
            LIMIT 1
            """,
            (BUILD_ID, chunk_hash, exclude_rag_chunk_id),
        ).fetchone()
    return row is not None

def _mark_rag_duplicate(row: sqlite3.Row):
    with get_db() as conn:
        conn.execute(
            """
            UPDATE rag_chunks
            SET status='duplicate_skipped',
                embedding_status='skipped',
                qdrant_status='duplicate_skipped',
                failure_stage=NULL,
                last_error=NULL,
                updated_at=?
            WHERE rag_chunk_id=?
            """,
            (utc_now(), row["rag_chunk_id"]),
        )

def _retry_metadata_if_needed(row: sqlite3.Row) -> Tuple[Dict[str, Any], Dict[str, Any], bool]:
    """Rebuild metadata on metadata-stage retries so updated mappings/config can fix a failure."""
    payload = json.loads(row["metadata_json"] or "{}")
    is_retry = row["status"] == "failed" and row["failure_stage"] == "metadata"

    if is_retry:
        chunk_text = reconstruct_rag_chunk_text(row)
        payload = enrich_chunk_metadata(
            row["document_id"], int(row["page"]), int(row["chunk_index"]), chunk_text
        )

    validation = validate_qdrant_metadata(payload)
    if not is_retry:
        return payload, validation, validation["valid"]

    attempt_number = int(row["metadata_attempt_count"]) + 1
    started = utc_now()
    with get_db() as conn:
        conn.execute(
            """
            UPDATE rag_chunks
            SET metadata_json=?,
                metadata_attempt_count=metadata_attempt_count+1,
                status=?,
                failure_stage=?,
                last_error=?,
                updated_at=?
            WHERE rag_chunk_id=?
            """,
            (
                json.dumps(payload, ensure_ascii=False),
                "metadata_validated" if validation["valid"] else "failed",
                None if validation["valid"] else "metadata",
                None if validation["valid"] else json.dumps(validation),
                utc_now(), row["rag_chunk_id"],
            ),
        )
    record_attempt(
        "rag_chunk", row["rag_chunk_id"], "metadata", attempt_number,
        "succeeded" if validation["valid"] else "failed",
        started,
        None if validation["valid"] else ValueError(json.dumps(validation)),
    )
    return payload, validation, validation["valid"]

def process_rag_batch(rows: Sequence[sqlite3.Row]):
    if not rows:
        return

    model = load_embedding_model()
    client = get_qdrant_client()

    work_rows: List[sqlite3.Row] = []
    texts: List[str] = []
    payloads: List[Dict[str, Any]] = []
    batch_hashes: Set[str] = set()

    for row in rows:
        payload, validation, valid = _retry_metadata_if_needed(row)
        if not valid:
            # Initial invalid metadata was already recorded in prepare_rag_chunks;
            # retry invalid metadata is recorded by _retry_metadata_if_needed.
            continue

        chunk_hash = payload["content_hash"]
        if chunk_hash in batch_hashes or chunk_hash_already_indexed(chunk_hash, row["rag_chunk_id"]):
            _mark_rag_duplicate(row)
            continue

        batch_hashes.add(chunk_hash)
        work_rows.append(row)
        texts.append(payload["text"])
        payloads.append(payload)

    if not work_rows:
        return

    # Crash-safety rule: keep status metadata_validated/failed until the expensive
    # operation completes. A kernel/node failure therefore leaves work selectable on rerun.
    embed_started = utc_now()
    try:
        vectors = model.encode(
            texts,
            batch_size=min(EMBED_BATCH_SIZE, len(texts)),
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False,
        )
    except Exception as exc:
        with get_db() as conn:
            for row in work_rows:
                conn.execute(
                    """
                    UPDATE rag_chunks
                    SET embedding_status='failed',
                        embedding_attempt_count=embedding_attempt_count+1,
                        status='failed',
                        failure_stage='embed',
                        last_error=?,
                        updated_at=?
                    WHERE rag_chunk_id=?
                    """,
                    (traceback.format_exc()[-4000:], utc_now(), row["rag_chunk_id"]),
                )
        for row in work_rows:
            record_attempt(
                "rag_chunk", row["rag_chunk_id"], "embed",
                int(row["embedding_attempt_count"]) + 1,
                "failed", embed_started, exc,
            )
        return

    # Persist the successful embedding attempt count, but leave status selectable until upsert succeeds.
    with get_db() as conn:
        for row in work_rows:
            conn.execute(
                """
                UPDATE rag_chunks
                SET embedding_status='embedded',
                    embedding_attempt_count=embedding_attempt_count+1,
                    failure_stage=NULL,
                    last_error=NULL,
                    updated_at=?
                WHERE rag_chunk_id=?
                """,
                (utc_now(), row["rag_chunk_id"]),
            )
    for row in work_rows:
        record_attempt(
            "rag_chunk", row["rag_chunk_id"], "embed",
            int(row["embedding_attempt_count"]) + 1,
            "succeeded", embed_started,
        )

    upsert_started = utc_now()
    try:
        points = [
            PointStruct(
                id=row["qdrant_point_id"],
                vector=vector.tolist(),
                payload=payload,
            )
            for row, vector, payload in zip(work_rows, vectors, payloads)
        ]

        client.upsert(
            collection_name=QDRANT_COLLECTION,
            points=points,
            wait=True,
        )
    except Exception as exc:
        with get_db() as conn:
            for row in work_rows:
                conn.execute(
                    """
                    UPDATE rag_chunks
                    SET qdrant_status='failed',
                        qdrant_attempt_count=qdrant_attempt_count+1,
                        status='failed',
                        failure_stage='qdrant_upsert',
                        last_error=?,
                        updated_at=?
                    WHERE rag_chunk_id=?
                    """,
                    (traceback.format_exc()[-4000:], utc_now(), row["rag_chunk_id"]),
                )
        for row in work_rows:
            record_attempt(
                "rag_chunk", row["rag_chunk_id"], "qdrant_upsert",
                int(row["qdrant_attempt_count"]) + 1,
                "failed", upsert_started, exc,
            )
        return

    # Deterministic point IDs make the upsert idempotent even if a node fails after
    # Qdrant commits but before SQLite records success.
    with get_db() as conn:
        for row in work_rows:
            conn.execute(
                """
                UPDATE rag_chunks
                SET qdrant_status='indexed',
                    qdrant_attempt_count=qdrant_attempt_count+1,
                    status='indexed',
                    failure_stage=NULL,
                    last_error=NULL,
                    updated_at=?
                WHERE rag_chunk_id=?
                """,
                (utc_now(), row["rag_chunk_id"]),
            )
    for row in work_rows:
        record_attempt(
            "rag_chunk", row["rag_chunk_id"], "qdrant_upsert",
            int(row["qdrant_attempt_count"]) + 1,
            "succeeded", upsert_started,
        )

def rag_index_pass(statuses: Sequence[str] = ("metadata_validated",)):
    placeholders = ",".join("?" for _ in statuses)
    with get_db() as conn:
        rows = conn.execute(
            f"""
            SELECT * FROM rag_chunks
            WHERE run_id=? AND status IN ({placeholders})
            ORDER BY document_id, page, chunk_index
            """,
            (RUN_ID, *statuses),
        ).fetchall()

    if not rows:
        print("No RAG chunks for this indexing pass.")
        return

    for start in tqdm(
        range(0, len(rows), QDRANT_UPSERT_BATCH_SIZE),
        desc="Embedding + Qdrant batches",
    ):
        batch = rows[start:start + QDRANT_UPSERT_BATCH_SIZE]
        process_rag_batch(batch)

def retry_failed_rag():
    for retry_idx in range(MAX_STAGE_RETRIES):
        with get_db() as conn:
            failed = conn.execute(
                """
                SELECT COUNT(*) FROM rag_chunks
                WHERE run_id=? AND status='failed'
                """,
                (RUN_ID,),
            ).fetchone()[0]

        if failed == 0:
            break

        print(f"RAG retry {retry_idx + 1}/{MAX_STAGE_RETRIES}: {failed} chunks")
        rag_index_pass(("failed",))

    with get_db() as conn:
        conn.execute(
            """
            UPDATE rag_chunks
            SET status='permanently_failed',
                qdrant_status=CASE
                    WHEN qdrant_status='indexed' THEN qdrant_status
                    ELSE 'permanently_failed'
                END,
                updated_at=?
            WHERE run_id=? AND status='failed'
            """,
            (utc_now(), RUN_ID),
        )

def finalize_document_rag_states():
    with get_db() as conn:
        docs = conn.execute(
            """
            SELECT document_id
            FROM run_documents
            WHERE run_id=? AND accepted=1
            """,
            (RUN_ID,),
        ).fetchall()

    for row in docs:
        doc_id = row["document_id"]
        with get_db() as conn:
            counts = {
                status: count
                for status, count in conn.execute(
                    """
                    SELECT status, COUNT(*)
                    FROM rag_chunks
                    WHERE run_id=? AND document_id=?
                    GROUP BY status
                    """,
                    (RUN_ID, doc_id),
                ).fetchall()
            }

        nonterminal = sum(
            counts.get(s, 0)
            for s in [
                "pending", "metadata_enriching", "metadata_validated",
                "embedding", "embedded", "qdrant_pending", "failed",
            ]
        )
        if nonterminal:
            continue

        terminal_success = counts.get("indexed", 0) + counts.get("duplicate_skipped", 0)
        perm_failed = counts.get("permanently_failed", 0)

        if terminal_success == 0 and perm_failed > 0:
            doc_status = "permanently_failed"
            rag_status = "permanently_failed"
            qdrant_status = "permanently_failed"
        else:
            doc_status = "indexed"
            rag_status = "complete"
            qdrant_status = "indexed"

        with get_db() as conn:
            conn.execute(
                """
                UPDATE run_documents
                SET document_status=?, rag_status=?, qdrant_status=?, updated_at=?
                WHERE run_id=? AND document_id=?
                """,
                (doc_status, rag_status, qdrant_status, utc_now(), RUN_ID, doc_id),
            )

print("Batch embedding/Qdrant pipeline ready.")

## 16. Terminal-state validation, manifest statistics, crop dictionary, and snapshot automation

In [ ]:
SOURCE_TERMINAL = {"extracted", "duplicate_skipped", "permanently_failed"}
DOC_TERMINAL = {"indexed", "rejected", "duplicate_skipped", "permanently_failed"}
Q_TERMINAL = {"succeeded", "skipped_early_stop", "permanently_failed"}
RAG_TERMINAL = {"indexed", "duplicate_skipped", "permanently_failed"}

def table_status_counts(table: str, status_field: str) -> Dict[str, int]:
    with get_db() as conn:
        rows = conn.execute(
            f"""
            SELECT {status_field}, COUNT(*) AS n
            FROM {table}
            WHERE run_id=?
            GROUP BY {status_field}
            """,
            (RUN_ID,),
        ).fetchall()
    return {row[0]: int(row[1]) for row in rows}

def assert_all_terminal():
    problems = []

    source_counts = table_status_counts("source_tasks", "status")
    doc_counts = table_status_counts("run_documents", "document_status")
    q_counts = table_status_counts("qualification_chunks", "status")
    rag_counts = table_status_counts("rag_chunks", "status")

    for name, counts, terminal in [
        ("source_tasks", source_counts, SOURCE_TERMINAL),
        ("run_documents", doc_counts, DOC_TERMINAL),
        ("qualification_chunks", q_counts, Q_TERMINAL),
        ("rag_chunks", rag_counts, RAG_TERMINAL),
    ]:
        nonterminal = {k: v for k, v in counts.items() if k not in terminal}
        if nonterminal:
            problems.append(f"{name}: {nonterminal}")

    if problems:
        raise RuntimeError("Non-terminal work remains:\n" + "\n".join(problems))

    return {
        "source_tasks": source_counts,
        "run_documents": doc_counts,
        "qualification_chunks": q_counts,
        "rag_chunks": rag_counts,
    }

def metadata_quality_stats() -> Dict[str, Any]:
    with get_db() as conn:
        rows = conn.execute(
            """
            SELECT status, metadata_json, failure_stage
            FROM rag_chunks
            WHERE run_id=?
            """,
            (RUN_ID,),
        ).fetchall()

    stats = Counter()
    missing_or_invalid = Counter()

    for row in rows:
        stats["rag_chunks_total"] += 1
        payload = json.loads(row["metadata_json"] or "{}")
        validation = validate_qdrant_metadata(payload)

        if validation["valid"]:
            stats["contract_valid"] += 1
        elif row["status"] == "permanently_failed" and row["failure_stage"] == "metadata":
            stats["contract_permanently_failed"] += 1

        if payload.get("location"):
            stats["location_present"] += 1
        if payload.get("hardiness_zone"):
            stats["hardiness_zone_present"] += 1
        if validation["ak_hi_hardiness_exception"]:
            stats["hardiness_zone_ak_hi_exception"] += 1
        if payload.get("title"):
            stats["title_present"] += 1

        if "month_year" in payload:
            stats["month_year_field_present"] += 1
        if payload.get("month_year"):
            stats["month_year_known"] += 1
        else:
            stats["month_year_unknown"] += 1

        if payload.get("title") and payload.get("month_year") and payload.get("hardiness_zone"):
            stats["priority_full_metadata"] += 1

        for field in validation["missing_fields"] + validation["invalid_fields"]:
            missing_or_invalid[field] += 1

    expected_stat_keys = [
        "rag_chunks_total",
        "contract_valid",
        "contract_permanently_failed",
        "location_present",
        "hardiness_zone_present",
        "hardiness_zone_ak_hi_exception",
        "title_present",
        "month_year_field_present",
        "month_year_known",
        "month_year_unknown",
        "priority_full_metadata",
    ]
    result = {key: int(stats.get(key, 0)) for key in expected_stat_keys}
    result["missing_or_invalid"] = {
        field: int(missing_or_invalid.get(field, 0))
        for field in REQUIRED_QDRANT_FIELDS
    }
    return result

def get_qdrant_version() -> str:
    try:
        r = requests.get(
            QDRANT_URL.rstrip("/") + "/",
            headers=qdrant_headers(),
            timeout=10,
        )
        if r.ok:
            data = r.json()
            return str(data.get("version") or data.get("title") or "")
    except Exception:
        pass
    return ""

def run_validation_queries() -> List[Dict[str, Any]]:
    if not VALIDATION_QUERIES:
        return []

    model = load_embedding_model()
    client = get_qdrant_client()
    results = []

    for query in VALIDATION_QUERIES:
        vector = model.encode(
            [query],
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False,
        )[0].tolist()

        points = client.query_points(
            collection_name=QDRANT_COLLECTION,
            query=vector,
            limit=3,
            with_payload=True,
        ).points

        results.append({
            "query": query,
            "hits": len(points),
            "top_titles": [
                (p.payload or {}).get("title", "")
                for p in points
            ],
        })

    return results

def file_sha256(path: Path) -> str:
    return bytes_sha256(path)

def completed_states_for_build() -> List[str]:
    with get_db() as conn:
        rows = conn.execute(
            """
            SELECT state_code
            FROM runs
            WHERE build_id=?
              AND sequence_number <= ?
              AND run_id != ?
              AND status='complete'
            ORDER BY sequence_number
            """,
            (BUILD_ID, RUN_SEQUENCE, RUN_ID),
        ).fetchall()
    states = [row["state_code"] for row in rows]
    if STATE_CODE not in states:
        states.append(STATE_CODE)
    return states

def run_statistics() -> Dict[str, Any]:
    with get_db() as conn:
        source_total = conn.execute(
            "SELECT COUNT(*) FROM source_tasks WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()[0]

        duplicate_count = conn.execute(
            """
            SELECT COUNT(*) FROM source_tasks
            WHERE run_id=? AND status='duplicate_skipped'
            """,
            (RUN_ID,),
        ).fetchone()[0]

        source_perm_failed = conn.execute(
            """
            SELECT COUNT(*) FROM source_tasks
            WHERE run_id=? AND status='permanently_failed'
            """,
            (RUN_ID,),
        ).fetchone()[0]

        rows = conn.execute(
            """
            SELECT
              SUM(CASE WHEN accepted=1 THEN 1 ELSE 0 END) AS accepted,
              SUM(CASE WHEN qualification_tag='msc' THEN 1 ELSE 0 END) AS rejected,
              SUM(CASE WHEN document_status='permanently_failed' THEN 1 ELSE 0 END) AS perm_failed
            FROM run_documents
            WHERE run_id=? AND duplicate=0
            """,
            (RUN_ID,),
        ).fetchone()

        q_total = conn.execute(
            """
            SELECT COUNT(*) FROM qualification_chunks
            WHERE run_id=? AND status='succeeded'
            """,
            (RUN_ID,),
        ).fetchone()[0]
        q_perm = conn.execute(
            """
            SELECT COUNT(*) FROM qualification_chunks
            WHERE run_id=? AND status='permanently_failed'
            """,
            (RUN_ID,),
        ).fetchone()[0]

        rag_created = conn.execute(
            "SELECT COUNT(*) FROM rag_chunks WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()[0]
        rag_indexed = conn.execute(
            """
            SELECT COUNT(*) FROM rag_chunks
            WHERE run_id=? AND status='indexed'
            """,
            (RUN_ID,),
        ).fetchone()[0]
        rag_duplicate = conn.execute(
            """
            SELECT COUNT(*) FROM rag_chunks
            WHERE run_id=? AND status='duplicate_skipped'
            """,
            (RUN_ID,),
        ).fetchone()[0]
        rag_perm = conn.execute(
            """
            SELECT COUNT(*) FROM rag_chunks
            WHERE run_id=? AND status='permanently_failed'
            """,
            (RUN_ID,),
        ).fetchone()[0]

        cumulative_docs = conn.execute(
            """
            SELECT COUNT(DISTINCT rd.document_id)
            FROM run_documents rd
            JOIN runs r ON r.run_id = rd.run_id
            WHERE r.build_id=?
              AND r.sequence_number <= ?
              AND rd.duplicate=0
            """,
            (BUILD_ID, RUN_SEQUENCE),
        ).fetchone()[0]

        cumulative_runs = conn.execute(
            """
            SELECT COUNT(*) FROM runs
            WHERE build_id=? AND (
                status='complete' OR run_id=?
            )
            """,
            (BUILD_ID, RUN_ID),
        ).fetchone()[0]

        cumulative_rag = conn.execute(
            """
            SELECT COUNT(*) FROM rag_chunks rc
            JOIN runs r ON r.run_id = rc.run_id
            WHERE r.build_id=?
              AND rc.status='indexed'
              AND r.sequence_number <= ?
            """,
            (BUILD_ID, RUN_SEQUENCE),
        ).fetchone()[0]

    return {
        "this_run": {
            "sources_discovered": int(source_total or 0),
            "duplicates_skipped": int(duplicate_count or 0),
            "sources_permanently_failed": int(source_perm_failed or 0),
            "documents_accepted": int(rows["accepted"] or 0),
            "documents_rejected": int(rows["rejected"] or 0),
            "documents_permanently_failed": int(rows["perm_failed"] or 0),
            "qualification_chunks_processed": int(q_total or 0),
            "qualification_chunks_permanently_failed": int(q_perm or 0),
            "rag_chunks_created": int(rag_created or 0),
            "rag_chunks_indexed": int(rag_indexed or 0),
            "rag_chunks_duplicate_skipped": int(rag_duplicate or 0),
            "rag_chunks_permanently_failed": int(rag_perm or 0),
        },
        "cumulative": {
            "states": int(cumulative_runs or 0),
            "documents": int(cumulative_docs or 0),
            "rag_chunks": int(cumulative_rag or 0),
            "qdrant_points": qdrant_point_count(),
        },
    }

def create_and_download_snapshot() -> Tuple[Path, Dict[str, Any]]:
    client = get_qdrant_client()
    snapshot = client.create_snapshot(collection_name=QDRANT_COLLECTION)

    snapshot_name = snapshot.name
    snapshot_path = STATE_DIR / f"mirage_base_{RUN_ID}.snapshot"

    with requests.get(
        f"{QDRANT_URL.rstrip('/')}/collections/{QDRANT_COLLECTION}/snapshots/{snapshot_name}",
        headers=qdrant_headers(),
        stream=True,
        timeout=None,
    ) as response:
        response.raise_for_status()
        with open(snapshot_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)

    info = {
        "server_snapshot_name": snapshot_name,
        "file": snapshot_path.name,
        "sha256": file_sha256(snapshot_path),
        "size_bytes": snapshot_path.stat().st_size,
        "qdrant_point_count": qdrant_point_count(),
    }
    return snapshot_path, info

def finalize_run_artifacts():
    terminal_counts = assert_all_terminal()

    with get_db() as conn:
        conn.execute(
            "UPDATE runs SET status='validating' WHERE run_id=?",
            (RUN_ID,),
        )

    qdrant_count = qdrant_point_count()
    validation_queries = run_validation_queries()
    metadata_quality = metadata_quality_stats()

    crop_dictionary = build_crop_dictionary_from_ledger()
    crop_dict_path = update_cumulative_crop_occurrences(crop_dictionary)

    stats = run_statistics()

    with get_db() as conn:
        conn.execute(
            "UPDATE runs SET status='snapshotting' WHERE run_id=?",
            (RUN_ID,),
        )

    snapshot_path, snapshot_info = create_and_download_snapshot()

    previous_run = None
    if PREVIOUS_RUN_DIR:
        previous_manifest = load_json(PREVIOUS_RUN_DIR / "manifest.json", {})
        previous_run = previous_manifest.get("run_id")

    manifest = {
        "schema_version": "1.0",
        "build_id": BUILD_ID,
        "run_id": RUN_ID,
        "sequence_number": RUN_SEQUENCE,
        "state_processed": {
            "name": STATE_NAME,
            "code": STATE_CODE,
        },
        "states_included": completed_states_for_build(),
        "previous_run": previous_run,
        "collection_name": QDRANT_COLLECTION,
        "versions": {
            "extractor": EXTRACTOR_VERSION,
            "classifier": CLASSIFIER_VERSION,
            "chunker": CHUNKER_VERSION,
            "metadata_contract": METADATA_CONTRACT_VERSION,
            "classifier_model": CLASSIFIER_MODEL_ID,
            "embedding_model": EMBEDDING_MODEL,
            "qdrant": get_qdrant_version(),
        },
        **stats,
        "metadata_quality": metadata_quality,
        "terminal_status_counts": terminal_counts,
        "validation_queries": validation_queries,
        "snapshot": snapshot_info,
        "crop_dictionary": {
            "file": crop_dict_path.name,
            "mode": "cumulative_single_file",
            "state_key": STATE_KEY,
            "crop_count": len(crop_dictionary.get(STATE_KEY, {})),
        },
        "started_at": None,
        "completed_at": utc_now(),
    }

    with get_db() as conn:
        run_row = conn.execute(
            "SELECT started_at FROM runs WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()
        manifest["started_at"] = run_row["started_at"]

    manifest_path = STATE_DIR / "manifest.json"
    write_json_atomic(manifest_path, manifest)

    snapshot_id = stable_hash(f"{BUILD_ID}|{RUN_ID}|{snapshot_info['sha256']}")
    with get_db() as conn:
        conn.execute(
            """
            INSERT OR REPLACE INTO snapshots(
                snapshot_id, build_id, run_id, collection_name,
                snapshot_path, manifest_path, checksum_sha256,
                qdrant_point_count, created_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                snapshot_id, BUILD_ID, RUN_ID, QDRANT_COLLECTION,
                str(snapshot_path), str(manifest_path),
                snapshot_info["sha256"], snapshot_info["qdrant_point_count"],
                utc_now(),
            ),
        )
        conn.execute(
            """
            UPDATE runs
            SET status='complete',
                completed_at=?,
                snapshot_path=?,
                manifest_path=?,
                error=NULL
            WHERE run_id=?
            """,
            (utc_now(), str(snapshot_path), str(manifest_path), RUN_ID),
        )

    print("Run finalized.")
    print("Snapshot:", snapshot_path)
    print("Manifest:", manifest_path)
    print("Cumulative crop occurrences updated:", crop_dict_path)

    return manifest

print("Validation/snapshot/manifest helpers ready.")

## 17. End-to-end orchestrator

This function is **resume-safe**. Each stage consults persisted SQLite state and deterministic IDs before doing work.

The classifier is unloaded before embedding to free GPU memory.

In [ ]:
def preflight_environment():
    configured_sources = bool(URL_FILE or PDF_DIR or PDF_ZIP_FILE or CSV_INPUTS)
    if not configured_sources:
        raise RuntimeError("No input sources are configured for this state run.")

    try:
        response = requests.get(
            QDRANT_URL.rstrip("/") + "/collections",
            headers=qdrant_headers(),
            timeout=10,
        )
        response.raise_for_status()
    except Exception as exc:
        raise RuntimeError(
            f"Qdrant is not reachable at {QDRANT_URL}. Start the Qdrant server "
            "on this compute node before launching the long run."
        ) from exc

    print("Preflight OK: Qdrant reachable and inputs configured.")

def print_run_status():
    with get_db() as conn:
        run = conn.execute(
            "SELECT * FROM runs WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()

    print(f"Run {RUN_ID}: {run['status']}")
    for table, field in [
        ("source_tasks", "status"),
        ("run_documents", "document_status"),
        ("qualification_chunks", "status"),
        ("rag_chunks", "status"),
    ]:
        try:
            print(f"{table}: {table_status_counts(table, field)}")
        except Exception as exc:
            print(f"{table}: {exc}")

def run_pipeline():
    # Fully idempotent completed-run behavior: do not create another snapshot on rerun.
    with get_db() as conn:
        existing_run = conn.execute(
            "SELECT status, manifest_path FROM runs WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()
    if existing_run and existing_run["status"] == "complete":
        manifest_path = Path(existing_run["manifest_path"] or (STATE_DIR / "manifest.json"))
        if not manifest_path.exists():
            raise RuntimeError(
                f"Run is marked complete but manifest is missing: {manifest_path}"
            )
        print(f"Run {RUN_ID} is already complete; returning existing manifest without reprocessing.")
        return load_json(manifest_path)

    preflight_environment()

    try:
        with get_db() as conn:
            conn.execute(
                """
                UPDATE runs
                SET status='running',
                    started_at=COALESCE(started_at, ?),
                    error=NULL
                WHERE run_id=?
                """,
                (utc_now(), RUN_ID),
            )

        print("\n=== 1/8 Discover sources ===")
        discover_sources()

        print("\n=== 2/8 Extract + canonicalize ===")
        extraction_pass(("discovered", "failed"))
        retry_failed_extraction()

        print("\n=== 3/8 Qualification initial pass ===")
        qualification_initial_pass()

        print("\n=== 4/8 Qualification retries + decisions ===")
        with get_db() as conn:
            conn.execute(
                "UPDATE runs SET status='retrying' WHERE run_id=?",
                (RUN_ID,),
            )
        retry_failed_qualification()
        finalize_qualification_decisions()

        # Qualification model is no longer needed; free GPU before embeddings.
        unload_classifier_model()

        print("\n=== 5/8 Prepare RAG chunks + metadata contract ===")
        prepare_rag_chunks()

        print("\n=== 6/8 Qdrant continuity + batch indexing ===")
        ensure_qdrant_continuity()
        rag_index_pass(("metadata_validated",))

        print("\n=== 7/8 RAG retries + document terminal states ===")
        retry_failed_rag()
        finalize_document_rag_states()

        print("\n=== 8/8 Validate + snapshot + manifest ===")
        manifest = finalize_run_artifacts()

        print("\n✓ COMPLETE:", RUN_ID)
        return manifest

    except Exception as exc:
        with get_db() as conn:
            conn.execute(
                "UPDATE runs SET status='failed', error=? WHERE run_id=?",
                (traceback.format_exc()[-8000:], RUN_ID),
            )
        print_run_status()
        raise

print("Orchestrator ready.")

## 18. Review configuration, then run

The default is `RUN_PIPELINE = False` so reopening the notebook cannot accidentally launch a large state build.

## Operational safety notes

- Keep `RUN_PIPELINE = False` while reviewing the configuration and run the status/preflight cells first.
- Qdrant must already be running and reachable at `QDRANT_URL`.
- For state run 2+, either keep the cumulative live collection running or set `PREVIOUS_RUN_DIR` to the immediately previous run folder so the notebook can restore its snapshot.
- The SQLite ledger and deterministic IDs make reruns idempotent. A failed/killed run resumes from persisted work rather than rebuilding completed stages.
- The notebook never resets a healthy cumulative collection just because a cell is rerun. Snapshot restoration only occurs when continuity validation requires it and `AUTO_RESTORE_PREVIOUS_SNAPSHOT=True`.

In [ ]:
print_run_status()

if RUN_PIPELINE:
    manifest = run_pipeline()
else:
    print(
        "\nRUN_PIPELINE is False. Review the configuration cell, "
        "set RUN_PIPELINE = True, and rerun this cell when ready."
    )

## 19. Recovery / inspection helpers

These are safe to run after interruption. The ledger is the source of truth for what remains.

In [ ]:
def show_failed_units(limit: int = 50):
    with get_db() as conn:
        sources = conn.execute(
            """
            SELECT source_key, source_type, source_uri, status, attempt_count, last_error
            FROM source_tasks
            WHERE run_id=? AND status IN ('failed','permanently_failed')
            LIMIT ?
            """,
            (RUN_ID, limit),
        ).fetchall()

        qchunks = conn.execute(
            """
            SELECT qualification_chunk_id, document_id, chunk_index,
                   status, attempt_count, last_error
            FROM qualification_chunks
            WHERE run_id=? AND status IN ('failed','permanently_failed')
            LIMIT ?
            """,
            (RUN_ID, limit),
        ).fetchall()

        rchunks = conn.execute(
            """
            SELECT rag_chunk_id, document_id, page, chunk_index,
                   status, failure_stage, metadata_attempt_count,
                   embedding_attempt_count, qdrant_attempt_count, last_error
            FROM rag_chunks
            WHERE run_id=? AND status IN ('failed','permanently_failed')
            LIMIT ?
            """,
            (RUN_ID, limit),
        ).fetchall()

    print("Source failures:")
    for row in sources:
        print(dict(row))

    print("\nQualification failures:")
    for row in qchunks:
        print(dict(row))

    print("\nRAG failures:")
    for row in rchunks:
        print(dict(row))

def show_metadata_quality():
    print(json.dumps(metadata_quality_stats(), indent=2))

def show_run_manifest():
    path = STATE_DIR / "manifest.json"
    if not path.exists():
        print("Manifest has not been created yet.")
        return
    print(path.read_text(encoding="utf-8"))

print("Inspection helpers ready.")

## 20. Notes

- The live Qdrant collection is cumulative; each state snapshot is also cumulative through that state.
- On a fresh Delta allocation for state 2+, set `PREVIOUS_RUN_DIR` so the prior snapshot can be restored automatically.
- The canonical store and SQLite ledger survive Qdrant-server restarts and allow the notebook to resume without repeating successful extraction/classification work.
- `month_year=""` is valid and simply means month-based priority filters will not apply to that chunk.
- AK/HI are the only explicit exception where `hardiness_zone=""` passes the metadata contract.
- The runtime database/inference mutation strategy is intentionally not handled here.